# Clickstream Sessionization Pipeline

Turns raw clickstream events into per-user sessions using event-time gaps. It handles
out-of-order events, duplicates, missing timestamps and boundary cases; records every run;
rejects bad rows with a reason instead of dropping them silently; and can be re-run safely.

Stack: Python, pandas, DuckDB (warehouse), pytest. Runs in Google Colab or locally.

## Quick start

**Google Colab (documented cloud environment)**

1. Open colab.research.google.com, then File > Upload notebook, and pick `sessionization_submission.ipynb`.
2. Runtime > Run all. The notebook writes the source files and tests, runs pytest, runs the
   pipeline on the sample, then on a generated 100k dataset, and prints the reports.
3. Colab erases its files when the runtime resets. Download the zip from the last cell to keep outputs.

**Locally**

    pip install pandas numpy duckdb pytest
    python -m pytest -q                      # all tests
    python -m pipeline.run                   # sample data -> output/warehouse.duckdb
    python -m pipeline.report                # reconciliation checks + reports
    python -m pipeline.generate --events 100000 --seed 42   # large dataset in data/large
    python -m pipeline.run --raw-dir data/large/raw --users data/large/ref/users.csv --db output/large.duckdb

`pipeline.run` exits with code 0 on success and 1 on failure; `--fail-at` injects a failure
(after_sessions, after_events or before_commit) to demonstrate rollback.

## Layout

    pipeline/
      config.py      settings: 30-minute gap, 1-day future tolerance, paths, fault flag
      ingest.py      read CSVs as raw strings, add lineage columns
      validate.py    9 reason codes, duplicate handling, UTC normalisation
      sessionize.py  gap rule, deterministic session_id
      publish.py     DuckDB tables, one transaction, late-event rebuild
      run_log.py     pipeline_runs table
      run.py         run_pipeline() and the command line
      reports.py     10 reconciliation checks, 7 reports (report.py is the CLI shim)
      synth.py       synthetic events with a known answer
      generate.py    seeded dataset generator
      evidence.py    compare / fingerprint / timing helpers
    tests/           pytest suites (one per module)
    data/            hand-built sample (ref, raw) and its expected answers
    docs/            DESIGN.md (original design), DECISIONS.md (what changed and why)

## Data model

Source (CSV): `raw_events` (event_id, user_id, event_ts, event_type, page_url) and
`users` (user_id, signup_ts, country). Each row also gets source_file, line_no (physical
line, header = 1), ingested_at and run_id.

Target (DuckDB):

| Table | Contents |
|---|---|
| events_clean | one row per valid event, with its session_id |
| sessions | session_id, user_id, start, end, duration_seconds, event_count, entry_page, exit_page |
| rejected_events | the raw row, reason_code, reason_detail, source_file, line_no, run_id |
| pipeline_runs | run_id, status (RUNNING/SUCCESS/FAILED), start/finish, row counts, error_message |

## Rules

Validation, in this order; each rejected row carries the first rule it fails:

| Reason code | Rule |
|---|---|
| MISSING_EVENT_ID / MISSING_USER_ID | blank |
| MISSING_TIMESTAMP | blank event_ts (rejected, not filled with ingestion time: a wrong time would silently corrupt session gaps) |
| INVALID_TIMESTAMP | not ISO-8601, or outside years 1677-2262 |
| FUTURE_TIMESTAMP | more than 1 day after ingestion |
| UNKNOWN_USER | user_id not in users |
| INVALID_EVENT_TYPE | not page_view, click, add_to_cart or purchase (case-sensitive) |
| DUPLICATE_EXACT / DUPLICATE_CONFLICT | same event_id; the earliest copy (ingested_at, source_file, line_no) is kept; CONFLICT means the payload differs |

Timestamps are converted to UTC; a timestamp with no timezone is treated as UTC.

Sessionization:

- Events are ordered per user by (event_ts, event_id); the event_id breaks ties between identical timestamps.
- A new session starts at a user's first event, or when the gap to the previous event is **more than** 30 minutes. A gap of **exactly** 30:00 stays in the same session; 30:01 starts a new one.
- Sessions are not split at midnight or at batch boundaries. A single-event session has duration 0.
- session_id is the first 16 hex characters of sha256(user_id + "|" + session_start in ISO format), so a rerun produces the same ids.

## Reliability

- **One transaction per publish.** Any failure rolls back every change; the run is then recorded as FAILED with the error, and the error is raised again. The run log is written outside that transaction so the FAILED row survives.
- **Safe reruns.** An event_id that is already stored is kept and counted as rows_already_loaded (not a reject). Rejects are replaced per source file. Rerunning the same input leaves all tables unchanged.
- **Late events.** When a later batch brings events for a user, that user's sessions are rebuilt from all their stored and new events and replaced in the same transaction. A late event can join two sessions or move a session's start.
- **Reconciliation.** rows_read = rows_clean + rows_rejected + rows_already_loaded is checked before publishing and again in the reports.

## Data quality and reports

`python -m pipeline.report` prints 10 SQL checks (each counts violations; 0 = PASS) and 7 reports:
session metrics, sessions per day, top entry pages, events by type, rejects by reason, rejects by file, and run history.
The checks include an independent proof of the gap rule: inside a session no gap exceeds 30:00,
and between two sessions of one user the gap is always more than 30:00.

Sample queries:

    SELECT reason_code, COUNT(*) AS n FROM rejected_events GROUP BY 1 ORDER BY 2 DESC;
    SELECT user_id, session_start, duration_seconds, event_count
    FROM sessions ORDER BY duration_seconds DESC LIMIT 5;

## Testing evidence

| Test file | Tests | What it proves |
|---|---|---|
| test_config | 7 | defaults and validation of settings |
| test_sample_data | 6 | the hand-built sample and its expected answers agree |
| test_ingest | 15 | raw values kept, line numbers, file-level failures |
| test_validate | 25 | every reason code, rule order, duplicates, UTC handling |
| test_sessionize | 15 | the 30:00 / 30:01 boundary, midnight, ties, out-of-order input, session ids |
| test_publish | 15 | idempotent rerun, late events, rollback at each failure point |
| test_run | 10 | run log, FAILED then successful retry, reconciliation guard, exit codes |
| test_generate | 10 | same seed gives identical files; pipeline reproduces the generator's known answer |
| test_reports | 16 | all checks pass on healthy data; each check fails when the matching damage is injected |
| test_evidence | 5 | comparison and fingerprint helpers |
| test_timestamps | 6 | bulk timestamp parser equals the careful per-value parser; out-of-range years rejected |

End-to-end results (Colab, seed 42):

- **Sample**: 22 rows -> 13 clean, 9 rejected (all 9 reason codes), 6 sessions; matches the hand-built expected files.
- **100,000 events** in two batches (main batch of 101,300 rows including 3,300 injected bad rows, then a late batch of 2,000 events): 100,000 clean events, 3,300 rejects, **25,079 sessions, identical to the generator's known answer** (0 missing, 0 unexpected). The late batch merged 396 sessions that had been split only because those events were missing.
- All 10 reconciliation checks PASS.
- **Rerun** of both batches: 0 new rows (98,000 and 2,000 already loaded); a fingerprint (row count plus MD5 of content) of all three tables is unchanged.

## Performance

Generated 100k events, Colab: main batch about 4.5-5.8 s, late batch about 1.3-1.8 s (run-to-run variation on Colab is around 30%).
The stage profile showed validation was the slowest stage (about 2.0 s of 4.5 s). One optimization with two parts,
each measured on the real 101,300 rows, best of 3, outputs asserted identical:

| Variant | validate() seconds | Speedup |
|---|---|---|
| original: per-value timestamp parse, sort every row for duplicates | 2.276 | 1.00x |
| bulk parse of canonical timestamps (strptime format) | 2.021 | 1.13x |
| bulk parse using pandas' ISO8601 parser | 1.855 | 1.23x |
| plus sort only repeated event_ids when flagging duplicates | 1.698 | 1.34x |

Only the stage speedup is claimed; whole-run timings vary by more than the gain.

## Assumptions

- All times are UTC; one CSV row is one event; processing is batch, not streaming.
- The users file is complete for the period processed.
- A stored event wins over a later copy with the same event_id.

## Known limitations and future work

- When late events arrive, all sessions of the affected users are rebuilt (2,000 late events rewrote 15,988 sessions). Correct, and fast enough here, but it scales with the affected users' history.
- Session ids are derived from the session start, so a late event that moves a session's start changes its id.
- validate sorts the clean rows and sessionize sorts them again; removing one sort would save about 0.6 s per 100k rows but changes the order validate returns rows in.
- DuckDB allows one writer at a time; there is no concurrent-run protection beyond that.
- Colab storage is temporary; download the zip to keep results.


## 1. Setup

In [1]:
!pip -q install pandas numpy duckdb pytest nbformat
import sys, platform
import pandas as pd, duckdb, pytest

print("Python :", sys.version.split()[0])
print("Pandas :", pd.__version__)
print("DuckDB :", duckdb.__version__)
print("Pytest :", pytest.__version__)
print("OS     :", platform.platform())

Python : 3.13.18
Pandas : 2.2.3
DuckDB : 1.3.2
Pytest : 8.4.2
OS     : Linux-6.6.159+-x86_64-with-glibc2.39


## 2. Project files

Each cell below writes one file, exactly as it was tested.

In [2]:
from pathlib import Path

for folder in ['data/expected', 'data/raw', 'data/ref', 'docs', 'output', 'pipeline', 'tests']:
    Path(folder).mkdir(parents=True, exist_ok=True)

In [3]:
%%writefile pytest.ini
[pytest]
pythonpath = .
testpaths = tests


Writing pytest.ini


In [4]:
%%writefile pipeline/__init__.py


Writing pipeline/__init__.py


In [5]:
%%writefile pipeline/config.py
"""Central configuration for the sessionization pipeline."""
from dataclasses import dataclass
from pathlib import Path

PROJECT_ROOT = Path(__file__).resolve().parents[1]

ALLOWED_EVENT_TYPES = ("page_view", "click", "add_to_cart", "purchase")


@dataclass(frozen=True)
class Config:
    session_gap_minutes: int = 30
    future_tolerance_days: int = 1
    allowed_event_types: tuple = ALLOWED_EVENT_TYPES
    raw_dir: Path = PROJECT_ROOT / "data" / "raw"
    users_path: Path = PROJECT_ROOT / "data" / "ref" / "users.csv"
    db_path: Path = PROJECT_ROOT / "output" / "warehouse.duckdb"
    # Fault injection: name of a publish step to fail at (None = no failure).
    fail_at: str | None = None

    def __post_init__(self):
        if self.session_gap_minutes <= 0:
            raise ValueError("session_gap_minutes must be > 0")
        if self.future_tolerance_days < 0:
            raise ValueError("future_tolerance_days must be >= 0")


Writing pipeline/config.py


In [6]:
%%writefile pipeline/evidence.py
"""Evidence helpers: compare warehouse output to a known answer, fingerprint tables, time stages."""
import time

import pandas as pd

from pipeline.config import Config
from pipeline.ingest import ingest_events, read_users
from pipeline.publish import connect, publish
from pipeline.sessionize import sessionize
from pipeline.validate import validate

COMPARE_COLUMNS = ["user_id", "session_start", "session_end", "duration_seconds", "event_count"]
TIME_FORMAT = "%Y-%m-%d %H:%M:%S"

# (order key, row expression) per table. run_id is left out on purpose: it changes every run.
FINGERPRINTS = {
    "events_clean": ("event_id",
                     "concat_ws('|', event_id, user_id, CAST(event_ts AS VARCHAR), event_type, "
                     "page_url, session_id)"),
    "sessions": ("session_id",
                 "concat_ws('|', session_id, user_id, CAST(session_start AS VARCHAR), "
                 "CAST(session_end AS VARCHAR), CAST(duration_seconds AS VARCHAR), "
                 "CAST(event_count AS VARCHAR), entry_page, exit_page)"),
    "rejected_events": ("source_file, line_no",
                        "concat_ws('|', source_file, CAST(line_no AS VARCHAR), event_id, "
                        "reason_code, reason_detail)"),
}


def compare_sessions(con, expected_csv) -> dict:
    """Compare the sessions table with an expected-sessions CSV (hand-built or generated)."""
    got = con.execute(f"SELECT {', '.join(COMPARE_COLUMNS)} FROM sessions").df()
    for col in ("session_start", "session_end"):
        got[col] = pd.to_datetime(got[col], utc=True).dt.strftime(TIME_FORMAT)
    got["duration_seconds"] = got["duration_seconds"].astype(str)
    got["event_count"] = got["event_count"].astype(str)
    want = pd.read_csv(expected_csv, dtype=str, keep_default_na=False)[COMPARE_COLUMNS]

    merged = got.merge(want, how="outer", on=COMPARE_COLUMNS, indicator=True)
    counts = merged["_merge"].value_counts()
    matched = int(counts.get("both", 0))
    return {"expected": len(want), "actual": len(got), "matched": matched,
            "missing": int(counts.get("right_only", 0)),
            "unexpected": int(counts.get("left_only", 0)),
            "ok": matched == len(want) == len(got)}


def fingerprint(con) -> dict:
    """{table: (row_count, md5 of the table's content)} - equal fingerprints mean equal tables."""
    return {table: con.execute(
                f"SELECT COUNT(*), md5(string_agg({row}, ',' ORDER BY {order})) FROM {table}"
            ).fetchone()
            for table, (order, row) in FINGERPRINTS.items()}


def time_stages(cfg: Config, run_id: str = "bench") -> pd.DataFrame:
    """Time each stage on cfg.raw_dir with a throwaway in-memory warehouse.

    publish includes a second sessionization (of the affected users), so stages overlap.
    """
    seconds = {}

    def stage(name, fn):
        start = time.perf_counter()
        result = fn()
        seconds[name] = time.perf_counter() - start
        return result

    raw = stage("ingest", lambda: ingest_events(cfg.raw_dir, run_id))
    users = read_users(cfg.users_path)
    clean, rejects = stage("validate", lambda: validate(raw, users, cfg))
    stage("sessionize", lambda: sessionize(clean, cfg, run_id))
    con = connect(":memory:")
    try:
        stage("publish", lambda: publish(con, clean, rejects,
                                         sorted(raw["source_file"].unique()), cfg, run_id))
    finally:
        con.close()

    out = pd.DataFrame({"stage": list(seconds), "seconds": list(seconds.values())})
    out["rows_per_second"] = (len(raw) / out["seconds"]).round(0)
    out["seconds"] = out["seconds"].round(3)
    return out


Writing pipeline/evidence.py


In [7]:
%%writefile pipeline/generate.py
"""Seeded generator for a large, messy clickstream dataset with a known answer.

Writes under out_dir:  ref/users.csv  raw/events_NNN.csv  late/late_000.csv
                       expected_sessions.csv  manifest.json

Usage:  python -m pipeline.generate [--out DIR] [--events N] [--users N] [--seed N]
                                    [--files N] [--late-fraction F]
"""
import argparse
import json
import shutil
from pathlib import Path

import numpy as np
import pandas as pd

from pipeline.config import PROJECT_ROOT
from pipeline.synth import BASE_TIME, RAW_COLUMNS, build_clean, user_ids

DEFAULT_RATES = {  # share of base events, one injected problem type each
    "DUPLICATE_EXACT": 0.010, "DUPLICATE_CONFLICT": 0.003, "MISSING_TIMESTAMP": 0.005,
    "INVALID_TIMESTAMP": 0.005, "FUTURE_TIMESTAMP": 0.002, "UNKNOWN_USER": 0.003,
    "MISSING_USER_ID": 0.001, "MISSING_EVENT_ID": 0.001, "INVALID_EVENT_TYPE": 0.003,
}
SCATTERED = [code for code in DEFAULT_RATES if not code.startswith("DUPLICATE")]
BAD_TIMESTAMPS = ["not-a-date", "2026-13-45T25:61:00Z", "yesterday", "12/31/2026"]
COUNTRIES = ["US", "DE", "IN", "GB", "FR"]


def _write_csv(df: pd.DataFrame, path: Path) -> int:
    df[RAW_COLUMNS].to_csv(path, index=False, lineterminator="\n")
    return len(df)


def _defects(rng, n_users, reason, count, tag) -> pd.DataFrame:
    """`count` otherwise-valid rows, each broken in exactly one way."""
    if count == 0:
        return pd.DataFrame(columns=RAW_COLUMNS)
    ts = BASE_TIME + pd.to_timedelta(rng.integers(0, 5 * 86400, count), unit="s")
    rows = pd.DataFrame({
        "event_id": [f"d{tag}_{i:06d}" for i in range(count)],
        "user_id": user_ids(rng.integers(0, n_users, count)),
        "event_ts": ts.strftime("%Y-%m-%dT%H:%M:%SZ"),
        "event_type": "click",
        "page_url": "/home",
    })
    if reason == "MISSING_TIMESTAMP":
        rows["event_ts"] = ""
    elif reason == "INVALID_TIMESTAMP":
        rows["event_ts"] = rng.choice(BAD_TIMESTAMPS, count)
    elif reason == "FUTURE_TIMESTAMP":
        rows["event_ts"] = "2099-01-01T00:00:00Z"
    elif reason == "UNKNOWN_USER":
        rows["user_id"] = [f"ghost{i:05d}" for i in range(count)]
    elif reason == "MISSING_USER_ID":
        rows["user_id"] = ""
    elif reason == "MISSING_EVENT_ID":
        rows["event_id"] = ""
    elif reason == "INVALID_EVENT_TYPE":
        rows["event_type"] = "scroll"
    return rows


def generate_dataset(out_dir, n_events=100_000, n_users=2_000, seed=42, n_files=5,
                     late_fraction=0.02, rates=None) -> dict:
    """Write the dataset and return the manifest (also saved as manifest.json)."""
    if n_events < 50 or n_users < 1 or n_files < 1:
        raise ValueError("need n_events >= 50, n_users >= 1 and n_files >= 1")
    out = Path(out_dir)
    for sub in ("raw", "late", "ref"):
        shutil.rmtree(out / sub, ignore_errors=True)
    (out / "raw").mkdir(parents=True)
    (out / "ref").mkdir(parents=True)

    rng = np.random.default_rng(seed)
    counts = {code: 0 if rate <= 0 else max(1, round(rate * n_events))
              for code, rate in {**DEFAULT_RATES, **(rates or {})}.items()}

    users = pd.DataFrame({
        "user_id": user_ids(np.arange(n_users)),
        "signup_ts": (BASE_TIME - pd.to_timedelta(rng.integers(30, 400, n_users), unit="D"))
                     .strftime("%Y-%m-%dT%H:%M:%SZ"),
        "country": rng.choice(COUNTRIES, n_users),
    })
    users.to_csv(out / "ref" / "users.csv", index=False, lineterminator="\n")

    events, truth, boundary = build_clean(rng, n_events, n_users)
    truth.to_csv(out / "expected_sessions.csv", index=False, lineterminator="\n")

    # Late batch: a random slice of the clean events, written to its own file.
    n_late = max(1, round(late_fraction * n_events)) if late_fraction > 0 else 0
    is_late = np.zeros(n_events, dtype=bool)
    is_late[rng.choice(n_events, n_late, replace=False)] = True
    main = events[~is_late].reset_index(drop=True)
    main["file_no"] = rng.integers(0, n_files, len(main))

    # Duplicates: copied from main events, placed in the same or a later file, after the original.
    n_exact, n_conflict = counts["DUPLICATE_EXACT"], counts["DUPLICATE_CONFLICT"]
    if n_exact + n_conflict > len(main):
        raise ValueError("dataset too small for the requested duplicates")
    dups = main.iloc[rng.choice(len(main), n_exact + n_conflict, replace=False)].copy()
    dups.loc[dups.index[n_exact:], "page_url"] = "/conflict"
    dups["file_no"] += (rng.random(len(dups)) * (n_files - dups["file_no"])).astype(int)

    scattered = pd.concat([_defects(rng, n_users, code, counts[code], i)
                           for i, code in enumerate(SCATTERED)], ignore_index=True)
    scattered["file_no"] = rng.integers(0, n_files, len(scattered))

    for f in range(n_files):
        body = pd.concat([main[main["file_no"] == f], scattered[scattered["file_no"] == f]],
                         ignore_index=True)
        body = body.iloc[rng.permutation(len(body))]  # out of order within the file
        _write_csv(pd.concat([body, dups[dups["file_no"] == f]]), out / "raw" / f"events_{f:03d}.csv")

    if n_late:
        (out / "late").mkdir()
        late = events[is_late]
        _write_csv(late.iloc[rng.permutation(len(late))], out / "late" / "late_000.csv")

    n_rejects = sum(counts.values())
    manifest = {
        "seed": seed, "n_users": n_users, "n_files": n_files,
        "base_events": n_events, "late_events": n_late, "sessions": len(truth),
        "injected": counts, "boundary_cases": boundary,
        "rows_written": {"main": len(main) + n_rejects, "late": n_late},
        "expected": {
            "main_run": {"rows_read": len(main) + n_rejects, "rows_clean": len(main),
                         "rows_rejected": n_rejects, "rows_already_loaded": 0},
            "late_run": ({"rows_read": n_late, "rows_clean": n_late,
                          "rows_rejected": 0, "rows_already_loaded": 0} if n_late else None),
            "final": {"events_clean": n_events, "sessions": len(truth),
                      "rejected_events": n_rejects},
        },
    }
    (out / "manifest.json").write_text(json.dumps(manifest, indent=2, sort_keys=True))
    return manifest


def main(argv=None) -> int:
    parser = argparse.ArgumentParser(description="Generate a large messy clickstream dataset.")
    parser.add_argument("--out", type=Path, default=PROJECT_ROOT / "data" / "large")
    parser.add_argument("--events", type=int, default=100_000)
    parser.add_argument("--users", type=int, default=2_000)
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument("--files", type=int, default=5)
    parser.add_argument("--late-fraction", type=float, default=0.02)
    a = parser.parse_args(argv)
    manifest = generate_dataset(a.out, a.events, a.users, a.seed, a.files, a.late_fraction)
    print(json.dumps(manifest, indent=2, sort_keys=True))
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


Writing pipeline/generate.py


In [8]:
%%writefile pipeline/ingest.py
"""Ingestion: read raw event CSVs exactly as written and add lineage columns."""
import csv
from pathlib import Path

import pandas as pd

EVENT_COLUMNS = ("event_id", "user_id", "event_ts", "event_type", "page_url")
LINEAGE_COLUMNS = ("source_file", "line_no", "ingested_at", "run_id")


class IngestionError(Exception):
    """File-level problem that should fail the whole run."""


def _as_utc(ts) -> pd.Timestamp:
    ts = pd.Timestamp(ts)
    return ts.tz_localize("UTC") if ts.tz is None else ts.tz_convert("UTC")


def read_events_file(path: Path) -> pd.DataFrame:
    """Read one CSV as raw strings. line_no is the physical line (header = 1)."""
    path = Path(path)
    with open(path, newline="", encoding="utf-8-sig") as f:
        reader = csv.DictReader(f)
        if not reader.fieldnames:
            raise IngestionError(f"{path.name}: file is empty (no header row)")
        reader.fieldnames = [h.strip() for h in reader.fieldnames]
        missing = [c for c in EVENT_COLUMNS if c not in reader.fieldnames]
        if missing:
            raise IngestionError(f"{path.name}: missing required columns {missing}")

        records = []
        for row in reader:
            record = {c: row.get(c) for c in EVENT_COLUMNS}
            record["line_no"] = reader.line_num
            records.append(record)

    df = pd.DataFrame(records, columns=[*EVENT_COLUMNS, "line_no"])
    df["source_file"] = path.name
    return df


def ingest_events(raw_dir: Path, run_id: str, ingested_at=None) -> pd.DataFrame:
    """Read every *.csv in raw_dir (sorted by name) and stamp lineage columns."""
    files = sorted(Path(raw_dir).glob("*.csv"))
    if not files:
        raise IngestionError(f"no .csv files found in {raw_dir}")

    df = pd.concat([read_events_file(p) for p in files], ignore_index=True)
    df["ingested_at"] = _as_utc(ingested_at if ingested_at is not None else pd.Timestamp.now(tz="UTC"))
    df["run_id"] = run_id
    return df[[*EVENT_COLUMNS, *LINEAGE_COLUMNS]]


def read_users(path: Path) -> pd.DataFrame:
    """Read the users reference file as strings."""
    path = Path(path)
    if not path.exists():
        raise IngestionError(f"users file not found: {path}")
    users = pd.read_csv(path, dtype=str, keep_default_na=False, encoding="utf-8-sig")
    if "user_id" not in users.columns:
        raise IngestionError(f"{path.name}: missing required column 'user_id'")
    return users


Writing pipeline/ingest.py


In [9]:
%%writefile pipeline/publish.py
"""Publish: write one validated batch into DuckDB in a single transaction.

Safe to rerun: events already stored are skipped, sessions are rebuilt only for
users who received new events, and rejects are replaced per source file.
"""
from dataclasses import dataclass
from pathlib import Path

import duckdb
import pandas as pd

from pipeline.config import Config
from pipeline.sessionize import SESSION_COLUMNS, sessionize
from pipeline.validate import CLEAN_COLUMNS, REJECT_COLUMNS

EVENT_COLUMNS = ["event_id", "user_id", "event_ts", "event_type", "page_url",
                 "session_id", "source_file", "ingested_at", "run_id"]
FAIL_POINTS = ("after_sessions", "after_events", "before_commit")

# No PRIMARY KEYs: DuckDB checks unique keys eagerly, so delete + re-insert of the
# same key in one transaction can fail. Uniqueness is guaranteed by the pipeline
# (dedup + whole-user replace) and verified by tests and reconciliation queries.
DDL = [
    """CREATE TABLE IF NOT EXISTS events_clean (
        event_id VARCHAR NOT NULL, user_id VARCHAR NOT NULL, event_ts TIMESTAMPTZ NOT NULL,
        event_type VARCHAR NOT NULL, page_url VARCHAR, session_id VARCHAR NOT NULL,
        source_file VARCHAR NOT NULL, ingested_at TIMESTAMPTZ NOT NULL, run_id VARCHAR NOT NULL)""",
    """CREATE TABLE IF NOT EXISTS sessions (
        session_id VARCHAR NOT NULL, user_id VARCHAR NOT NULL,
        session_start TIMESTAMPTZ NOT NULL, session_end TIMESTAMPTZ NOT NULL,
        duration_seconds BIGINT NOT NULL, event_count INTEGER NOT NULL,
        entry_page VARCHAR, exit_page VARCHAR, run_id VARCHAR NOT NULL)""",
    """CREATE TABLE IF NOT EXISTS rejected_events (
        event_id VARCHAR, user_id VARCHAR, event_ts VARCHAR, event_type VARCHAR, page_url VARCHAR,
        source_file VARCHAR NOT NULL, line_no INTEGER NOT NULL,
        reason_code VARCHAR NOT NULL, reason_detail VARCHAR, run_id VARCHAR NOT NULL)""",
]


class InjectedFailure(RuntimeError):
    """Raised on purpose by Config(fail_at=...) to prove rollback works."""


@dataclass
class PublishResult:
    rows_clean: int           # new events inserted this run
    rows_already_loaded: int  # valid events whose event_id was already stored
    rows_rejected: int
    sessions_written: int
    users_affected: int


def connect(db_path) -> duckdb.DuckDBPyConnection:
    if str(db_path) != ":memory:":
        Path(db_path).parent.mkdir(parents=True, exist_ok=True)
    con = duckdb.connect(str(db_path))
    con.execute("SET TimeZone = 'UTC'")
    for statement in DDL:
        con.execute(statement)
    return con


def _checkpoint(cfg: Config, name: str) -> None:
    if cfg.fail_at == name:
        raise InjectedFailure(f"injected failure at '{name}'")


def _insert(con, table: str, df: pd.DataFrame, columns: list) -> None:
    if df.empty:
        return
    cols = ", ".join(columns)
    con.register("_rows", df[columns])
    con.execute(f"INSERT INTO {table} ({cols}) SELECT {cols} FROM _rows")
    con.unregister("_rows")


def _stored_event_ids(con, clean: pd.DataFrame) -> set:
    if clean.empty:
        return set()
    con.register("_batch_ids", clean[["event_id"]])
    rows = con.execute(
        "SELECT event_id FROM events_clean WHERE event_id IN (SELECT event_id FROM _batch_ids)").fetchall()
    return {r[0] for r in rows}


def _publish(con, clean, rejects, source_files, cfg, run_id) -> PublishResult:
    new = clean[~clean["event_id"].isin(_stored_event_ids(con, clean))]
    affected = sorted(new["user_id"].unique())
    sessions_written = 0

    if affected:
        con.register("_affected", pd.DataFrame({"user_id": affected}))
        existing = con.execute(
            f"SELECT {', '.join(CLEAN_COLUMNS)} FROM events_clean "
            "WHERE user_id IN (SELECT user_id FROM _affected)").df()
        frames = [f for f in (existing, new[CLEAN_COLUMNS]) if not f.empty]
        events, sessions = sessionize(pd.concat(frames, ignore_index=True), cfg, run_id)

        con.execute("DELETE FROM sessions WHERE user_id IN (SELECT user_id FROM _affected)")
        _insert(con, "sessions", sessions, SESSION_COLUMNS)
        _checkpoint(cfg, "after_sessions")

        con.execute("DELETE FROM events_clean WHERE user_id IN (SELECT user_id FROM _affected)")
        _insert(con, "events_clean", events, EVENT_COLUMNS)
        _checkpoint(cfg, "after_events")
        sessions_written = len(sessions)

    if source_files:
        con.register("_files", pd.DataFrame({"source_file": list(source_files)}))
        con.execute("DELETE FROM rejected_events WHERE source_file IN (SELECT source_file FROM _files)")
    _insert(con, "rejected_events", rejects, REJECT_COLUMNS)

    return PublishResult(
        rows_clean=len(new),
        rows_already_loaded=len(clean) - len(new),
        rows_rejected=len(rejects),
        sessions_written=sessions_written,
        users_affected=len(affected),
    )


def publish(con, clean, rejects, source_files, cfg: Config, run_id: str) -> PublishResult:
    """All-or-nothing: any error rolls back every change made by this call."""
    con.begin()
    try:
        result = _publish(con, clean, rejects, source_files, cfg, run_id)
        _checkpoint(cfg, "before_commit")
        con.commit()
        return result
    except Exception:
        con.rollback()
        raise


Writing pipeline/publish.py


In [10]:
%%writefile pipeline/report.py
"""Shim so the report runs as:  python -m pipeline.report"""
from pipeline.reports import main

if __name__ == "__main__":
    raise SystemExit(main())


Writing pipeline/report.py


In [11]:
%%writefile pipeline/reports.py
"""Reports: reconciliation checks plus data-quality and session metric queries.

Usage:  python -m pipeline.report [--db FILE] [--gap-minutes N]   (exit 1 if a check fails)
"""
import argparse
from pathlib import Path

import pandas as pd

from pipeline.config import Config
from pipeline.publish import connect
from pipeline.run_log import ensure_runs_table

REASON_CODES = (
    "MISSING_EVENT_ID", "MISSING_USER_ID", "MISSING_TIMESTAMP", "INVALID_TIMESTAMP",
    "FUTURE_TIMESTAMP", "UNKNOWN_USER", "INVALID_EVENT_TYPE",
    "DUPLICATE_EXACT", "DUPLICATE_CONFLICT",
)

# Each event next to the previous event of the same user (same order the pipeline uses).
ORDERED = """
WITH ordered AS (
    SELECT session_id,
           epoch(event_ts) - epoch(LAG(event_ts) OVER w) AS gap_s,
           LAG(session_id) OVER w AS prev_session
    FROM events_clean
    WINDOW w AS (PARTITION BY user_id ORDER BY event_ts, event_id)
)"""

# Every check counts violations; 0 = PASS. {gap} is the threshold in seconds.
CHECKS = {
    "run_counts_reconcile": """SELECT COUNT(*) FROM pipeline_runs WHERE status = 'SUCCESS'
        AND rows_read <> rows_clean + rows_rejected + rows_already_loaded""",
    "no_runs_stuck_running": "SELECT COUNT(*) FROM pipeline_runs WHERE status = 'RUNNING'",
    "event_id_unique": """SELECT COUNT(*) FROM (
        SELECT event_id FROM events_clean GROUP BY event_id HAVING COUNT(*) > 1)""",
    "session_id_unique": """SELECT COUNT(*) FROM (
        SELECT session_id FROM sessions GROUP BY session_id HAVING COUNT(*) > 1)""",
    "events_have_session": """SELECT COUNT(*) FROM events_clean e
        LEFT JOIN sessions s ON e.session_id = s.session_id WHERE s.session_id IS NULL""",
    "sessions_have_events": """SELECT COUNT(*) FROM sessions s
        WHERE NOT EXISTS (SELECT 1 FROM events_clean e WHERE e.session_id = s.session_id)""",
    "session_summary_matches_events": """SELECT COUNT(*) FROM sessions s
        JOIN (SELECT session_id, COUNT(*) AS n, MIN(event_ts) AS lo, MAX(event_ts) AS hi
              FROM events_clean GROUP BY session_id) e ON e.session_id = s.session_id
        WHERE s.event_count <> e.n OR s.session_start <> e.lo OR s.session_end <> e.hi
           OR s.duration_seconds <> CAST(epoch(e.hi) - epoch(e.lo) AS BIGINT)""",
    "gap_inside_session_within_threshold":
        ORDERED + " SELECT COUNT(*) FROM ordered WHERE prev_session = session_id AND gap_s > {gap}",
    "gap_between_sessions_over_threshold":
        ORDERED + " SELECT COUNT(*) FROM ordered WHERE prev_session <> session_id AND gap_s <= {gap}",
    "reject_reasons_known":
        "SELECT COUNT(*) FROM rejected_events WHERE reason_code NOT IN ({codes})",
}

REPORTS = {
    "session_metrics": """SELECT COUNT(*) AS sessions, COUNT(DISTINCT user_id) AS users,
        ROUND(AVG(event_count), 2) AS avg_events,
        ROUND(AVG(duration_seconds), 1) AS avg_duration_s,
        median(duration_seconds) AS median_duration_s,
        quantile_cont(duration_seconds, 0.95) AS p95_duration_s,
        MAX(duration_seconds) AS max_duration_s,
        ROUND(100.0 * COUNT(*) FILTER (WHERE event_count = 1) / COUNT(*), 1) AS pct_single_event
        FROM sessions""",
            "sessions_per_day": """SELECT strftime(session_start AT TIME ZONE 'UTC', '%Y-%m-%d') AS day,
        COUNT(*) AS sessions, CAST(SUM(event_count) AS BIGINT) AS events
        FROM sessions GROUP BY 1 ORDER BY 1""",
    "top_entry_pages": """SELECT COALESCE(entry_page, '(none)') AS entry_page, COUNT(*) AS sessions
        FROM sessions GROUP BY 1 ORDER BY 2 DESC, 1 LIMIT 5""",
    "events_by_type": """SELECT event_type, COUNT(*) AS events
        FROM events_clean GROUP BY 1 ORDER BY 2 DESC, 1""",
    "dq_by_reason": """SELECT reason_code, COUNT(*) AS row_count,
        ROUND(100.0 * COUNT(*) / ((SELECT COUNT(*) FROM events_clean)
                                  + (SELECT COUNT(*) FROM rejected_events)), 2) AS pct_of_rows
        FROM rejected_events GROUP BY 1 ORDER BY 2 DESC, 1""",
    "dq_by_file": """WITH c AS (SELECT source_file, COUNT(*) AS n FROM events_clean GROUP BY 1),
        r AS (SELECT source_file, COUNT(*) AS n FROM rejected_events GROUP BY 1)
        SELECT COALESCE(c.source_file, r.source_file) AS source_file,
               COALESCE(c.n, 0) AS clean_rows, COALESCE(r.n, 0) AS rejected_rows,
               ROUND(100.0 * COALESCE(r.n, 0) / (COALESCE(c.n, 0) + COALESCE(r.n, 0)), 2) AS pct_rejected
        FROM c FULL OUTER JOIN r ON c.source_file = r.source_file ORDER BY 1""",
    "run_history": """SELECT run_id, status, rows_read, rows_clean, rows_rejected, rows_already_loaded,
        sessions_written, users_affected, ROUND(epoch(finished_at) - epoch(started_at), 2) AS seconds,
        error_message FROM pipeline_runs ORDER BY started_at""",
}


def run_checks(con, gap_minutes: int = 30) -> pd.DataFrame:
    """Run every check; returns one row per check with its violation count and PASS/FAIL."""
    params = {"gap": gap_minutes * 60, "codes": ", ".join(f"'{c}'" for c in REASON_CODES)}
    rows = []
    for name, sql in CHECKS.items():
        violations = con.execute(sql.format(**params)).fetchone()[0]
        rows.append({"check": name, "violations": violations,
                     "status": "PASS" if violations == 0 else "FAIL"})
    return pd.DataFrame(rows)


def run_report(con, name: str) -> pd.DataFrame:
    return con.execute(REPORTS[name]).df()


def main(argv=None) -> int:
    cfg = Config()
    parser = argparse.ArgumentParser(description="Print reconciliation checks and reports.")
    parser.add_argument("--db", type=Path, default=cfg.db_path)
    parser.add_argument("--gap-minutes", type=int, default=cfg.session_gap_minutes)
    args = parser.parse_args(argv)
    if not args.db.exists():
        print(f"warehouse not found: {args.db}")
        return 1

    con = connect(args.db)
    try:
        ensure_runs_table(con)
        checks = run_checks(con, args.gap_minutes)
        print("== reconciliation checks ==")
        print(checks.to_string(index=False))
        for name in REPORTS:
            print(f"\n== {name} ==")
            print(run_report(con, name).to_string(index=False))
    finally:
        con.close()

    failed = int((checks.status == "FAIL").sum())
    print("\nALL CHECKS PASSED" if failed == 0 else f"\n{failed} CHECK(S) FAILED")
    return 1 if failed else 0


if __name__ == "__main__":
    raise SystemExit(main())


Writing pipeline/reports.py


In [12]:
%%writefile pipeline/run.py
"""Entry point: ingest -> validate -> publish (incl. sessionize), recorded in pipeline_runs.

Usage:  python -m pipeline.run [--raw-dir DIR] [--users FILE] [--db FILE]
                               [--gap-minutes N] [--fail-at POINT]
"""
import argparse
import uuid
from dataclasses import asdict, dataclass
from pathlib import Path

import pandas as pd

from pipeline.config import Config
from pipeline.ingest import ingest_events, read_users
from pipeline.publish import FAIL_POINTS, connect, publish
from pipeline.run_log import ensure_runs_table, finish_run, start_run
from pipeline.validate import validate


class ReconciliationError(RuntimeError):
    """clean + rejected rows did not add up to rows read; nothing is published."""


@dataclass
class RunResult:
    run_id: str
    status: str
    rows_read: int
    rows_clean: int
    rows_rejected: int
    rows_already_loaded: int
    sessions_written: int
    users_affected: int


def new_run_id() -> str:
    return f"run_{pd.Timestamp.now(tz='UTC'):%Y%m%dT%H%M%S}_{uuid.uuid4().hex[:6]}"


def run_pipeline(cfg: Config | None = None, run_id: str | None = None) -> RunResult:
    """Run one batch. On failure the run is logged as FAILED and the error is re-raised."""
    cfg = cfg or Config()
    run_id = run_id or new_run_id()
    con = connect(cfg.db_path)
    try:
        ensure_runs_table(con)
        start_run(con, run_id)
        counts = {}
        try:
            raw = ingest_events(cfg.raw_dir, run_id)
            counts["rows_read"] = len(raw)
            clean, rejects = validate(raw, read_users(cfg.users_path), cfg)
            if len(clean) + len(rejects) != len(raw):
                raise ReconciliationError(
                    f"read {len(raw)} rows but clean={len(clean)} + rejected={len(rejects)}")
            result = publish(con, clean, rejects, sorted(raw["source_file"].unique()), cfg, run_id)
            counts.update(asdict(result))
        except Exception as exc:
            finish_run(con, run_id, "FAILED", counts, f"{type(exc).__name__}: {exc}")
            raise
        finish_run(con, run_id, "SUCCESS", counts)
        return RunResult(run_id=run_id, status="SUCCESS", **counts)
    finally:
        con.close()


def main(argv=None) -> int:
    parser = argparse.ArgumentParser(description="Sessionize raw clickstream events into DuckDB.")
    parser.add_argument("--raw-dir", type=Path)
    parser.add_argument("--users", type=Path, dest="users_path")
    parser.add_argument("--db", type=Path, dest="db_path")
    parser.add_argument("--gap-minutes", type=int, dest="session_gap_minutes")
    parser.add_argument("--fail-at", choices=FAIL_POINTS)
    args = parser.parse_args(argv)

    cfg = Config(**{k: v for k, v in vars(args).items() if v is not None})
    try:
        result = run_pipeline(cfg)
    except Exception as exc:
        print(f"Run FAILED: {type(exc).__name__}: {exc}")
        return 1
    print(f"Run SUCCESS: {result}")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


Writing pipeline/run.py


In [13]:
%%writefile pipeline/run_log.py
"""Run log: one row per pipeline run in pipeline_runs (RUNNING -> SUCCESS / FAILED)."""
import pandas as pd

COUNT_COLUMNS = ("rows_read", "rows_clean", "rows_rejected", "rows_already_loaded",
                 "sessions_written", "users_affected")

RUNS_DDL = """CREATE TABLE IF NOT EXISTS pipeline_runs (
    run_id VARCHAR NOT NULL, started_at TIMESTAMPTZ NOT NULL, finished_at TIMESTAMPTZ,
    status VARCHAR NOT NULL,
    rows_read INTEGER, rows_clean INTEGER, rows_rejected INTEGER,
    rows_already_loaded INTEGER, sessions_written INTEGER, users_affected INTEGER,
    error_message VARCHAR)"""


def _now():
    return pd.Timestamp.now(tz="UTC").to_pydatetime()


def ensure_runs_table(con) -> None:
    con.execute(RUNS_DDL)


def start_run(con, run_id: str) -> None:
    """Record a RUNNING row. Refuses to reuse a run_id so history is never overwritten."""
    if con.execute("SELECT 1 FROM pipeline_runs WHERE run_id = ?", [run_id]).fetchone():
        raise ValueError(f"run_id '{run_id}' already exists in pipeline_runs")
    con.execute("INSERT INTO pipeline_runs (run_id, started_at, status) VALUES (?, ?, 'RUNNING')",
                [run_id, _now()])


def finish_run(con, run_id: str, status: str, counts: dict, error_message: str | None = None) -> None:
    """Close the run. Counts not reached before a failure stay NULL."""
    assignments = ", ".join(f"{c} = ?" for c in COUNT_COLUMNS)
    con.execute(
        f"UPDATE pipeline_runs SET finished_at = ?, status = ?, error_message = ?, {assignments} "
        "WHERE run_id = ?",
        [_now(), status, error_message, *(counts.get(c) for c in COUNT_COLUMNS), run_id])


Writing pipeline/run_log.py


In [14]:
%%writefile pipeline/sessionize.py
"""Sessionization: group each user's clean events into sessions by event-time gaps."""
import hashlib

import pandas as pd

from pipeline.config import Config

SESSION_COLUMNS = ["session_id", "user_id", "session_start", "session_end",
                   "duration_seconds", "event_count", "entry_page", "exit_page", "run_id"]


def make_session_id(user_id: str, session_start: pd.Timestamp) -> str:
    """Deterministic id: same user + same start => same id on every run."""
    key = f"{user_id}|{session_start.isoformat()}"
    return hashlib.sha256(key.encode("utf-8")).hexdigest()[:16]


def sessionize(clean: pd.DataFrame, cfg: Config, run_id: str):
    """Return (events with session_id, sessions). Input must be validated clean events."""
    if clean.empty:
        return clean.assign(session_id=pd.Series(dtype=object)), pd.DataFrame(columns=SESSION_COLUMNS)

    events = clean.sort_values(["user_id", "event_ts", "event_id"]).reset_index(drop=True)
    gap = events.groupby("user_id")["event_ts"].diff()
    is_new = gap.isna() | (gap > pd.Timedelta(minutes=cfg.session_gap_minutes))
    events["_session_no"] = is_new.cumsum()

    # Take entry/exit from the real first/last row (groupby first/last would skip null pages).
    entry = events[is_new].set_index("_session_no")
    exit_ = events.drop_duplicates("_session_no", keep="last").set_index("_session_no")
    counts = events.groupby("_session_no").size()

    sessions = pd.DataFrame({
        "user_id": entry["user_id"],
        "session_start": entry["event_ts"],
        "session_end": exit_["event_ts"],
        "event_count": counts,
        "entry_page": entry["page_url"],
        "exit_page": exit_["page_url"],
    })
    sessions["duration_seconds"] = (
        (sessions["session_end"] - sessions["session_start"]).dt.total_seconds().astype("int64"))
    sessions["session_id"] = [make_session_id(u, s)
                              for u, s in zip(sessions["user_id"], sessions["session_start"])]
    sessions["run_id"] = run_id

    events["session_id"] = events["_session_no"].map(sessions["session_id"])
    events = events.drop(columns="_session_no")
    return events, sessions.reset_index(drop=True)[SESSION_COLUMNS]


Writing pipeline/sessionize.py


In [15]:
%%writefile pipeline/synth.py
"""Synthetic clean events with a known answer: sessions are laid out first, events second.

Inside a session every gap is at most 30:00; between two sessions of one user the
gap is at least 30:01. So the true sessions are known without running the pipeline.
"""
import numpy as np
import pandas as pd

from pipeline.config import ALLOWED_EVENT_TYPES

BASE_TIME = pd.Timestamp("2026-01-01", tz="UTC")
RAW_COLUMNS = ["event_id", "user_id", "event_ts", "event_type", "page_url"]
PAGES = ["/home", "/search", "/cart", "/checkout", "/account", "/help",
         "/product/1", "/product/2", "/product/3", "/product/4"]
TS_FORM_WEIGHTS = [0.85, 0.05, 0.04, 0.03, 0.03]
EVENT_TYPE_WEIGHTS = [0.60, 0.25, 0.10, 0.05]  # same order as ALLOWED_EVENT_TYPES


def user_ids(idx) -> np.ndarray:
    return np.char.add("u", np.char.zfill(np.asarray(idx).astype(str), 5))


def format_timestamps(rng, ts: pd.DatetimeIndex) -> np.ndarray:
    """Write each instant in one of five formats; all parse back to the same UTC value."""
    naive = ts.tz_localize(None)

    def iso(t):
        return pd.Series(t.strftime("%Y-%m-%dT%H:%M:%S"), dtype=object)

    forms = [
        iso(naive) + "Z",
        pd.Series(naive.strftime("%Y-%m-%d %H:%M:%S"), dtype=object),  # no timezone = UTC
        iso(naive) + "+00:00",
        iso(naive + pd.Timedelta(hours=2)) + "+02:00",
        iso(naive - pd.Timedelta(hours=5)) + "-05:00",
    ]
    pick = rng.choice(len(forms), size=len(ts), p=TS_FORM_WEIGHTS)
    stacked = np.stack([f.to_numpy() for f in forms])
    return stacked[pick, np.arange(len(ts))]


def build_clean(rng, n_events: int, n_users: int):
    """Return (events, true_sessions, boundary_counts) with exactly n_events events."""
    k = rng.geometric(0.25, size=n_events)  # events per session, mean 4
    cum = np.cumsum(k)
    last = int(np.searchsorted(cum, n_events))
    k = k[: last + 1].copy()
    k[-1] -= cum[last] - n_events  # trim the final session to hit n_events exactly
    n_sessions = len(k)

    session_no = np.repeat(np.arange(n_sessions), k)
    first_idx = np.cumsum(k) - k
    is_first = np.zeros(n_events, dtype=bool)
    is_first[first_idx] = True

    # Gaps between events inside a session: never more than 30:00.
    gap = rng.integers(1, 1741, n_events)
    roll = rng.random(n_events)
    gap[roll < 0.03] = 0                        # identical timestamps
    gap[(roll >= 0.03) & (roll < 0.05)] = 1800  # exactly the threshold: same session
    gap[is_first] = 0
    cs = np.cumsum(gap)
    offset = cs - np.repeat(cs[first_idx], k)
    duration = offset[first_idx + k - 1]

    # Gaps between sessions of one user: always more than 30:00.
    user_of_session = rng.integers(0, n_users, n_sessions)
    break_gap = rng.integers(1801, 21601, n_sessions)
    break_gap[rng.random(n_sessions) < 0.05] = 1801  # 30:01: just over the threshold
    order = np.argsort(user_of_session, kind="stable")
    user_sorted = user_of_session[order]
    block = duration[order] + break_gap[order]
    excl = np.cumsum(block) - block
    user_first = np.flatnonzero(np.r_[True, user_sorted[1:] != user_sorted[:-1]])
    group_size = np.diff(np.r_[user_first, n_sessions])
    rel_start = excl - np.repeat(excl[user_first], group_size)
    origin = rng.integers(0, 2 * 86400, n_users)
    start = np.empty(n_sessions, dtype=np.int64)
    start[order] = origin[user_sorted] + rel_start

    is_last_of_user = np.r_[user_sorted[1:] != user_sorted[:-1], True]
    boundary = {
        "same_session_gap_exactly_1800s": int((gap == 1800).sum()),
        "new_session_gap_exactly_1801s": int(((break_gap[order] == 1801) & ~is_last_of_user).sum()),
        "identical_timestamp_pairs": int(((gap == 0) & ~is_first).sum()),
    }

    ts = BASE_TIME + pd.to_timedelta(start[session_no] + offset, unit="s")
    event_ids = np.char.add("e", np.char.zfill(rng.permutation(n_events).astype(str), 8))
    events = pd.DataFrame({
        "event_id": event_ids,
        "user_id": user_ids(user_of_session[session_no]),
        "event_ts": format_timestamps(rng, ts),
        "event_type": rng.choice(ALLOWED_EVENT_TYPES, n_events, p=EVENT_TYPE_WEIGHTS),
        "page_url": np.where(rng.random(n_events) < 0.03, "", rng.choice(PAGES, n_events)),
    })

    truth = pd.DataFrame({
        "user_id": user_ids(user_of_session),
        "session_start": BASE_TIME + pd.to_timedelta(start, unit="s"),
        "session_end": BASE_TIME + pd.to_timedelta(start + duration, unit="s"),
        "duration_seconds": duration,
        "event_count": k,
    }).sort_values(["user_id", "session_start"]).reset_index(drop=True)
    for c in ("session_start", "session_end"):
        truth[c] = truth[c].dt.strftime("%Y-%m-%d %H:%M:%S")
    return events, truth, boundary


Writing pipeline/synth.py


In [16]:
%%writefile pipeline/validate.py
"""Validation: split ingested rows into clean events and rejects with a reason."""
from datetime import datetime

import pandas as pd

from pipeline.config import Config

TEXT_COLUMNS = ["event_id", "user_id", "event_ts", "event_type", "page_url"]
PAYLOAD_COLUMNS = ["user_id", "_ts", "event_type", "page_url"]  # compared for duplicates
CLEAN_COLUMNS = ["event_id", "user_id", "event_ts", "event_type", "page_url",
                 "source_file", "ingested_at", "run_id"]
REJECT_COLUMNS = [*TEXT_COLUMNS, "source_file", "line_no", "reason_code", "reason_detail", "run_id"]

# Canonical 'YYYY-MM-DDTHH:MM:SSZ' with every digit range-checked; parsed in bulk.
CANONICAL_TS = (r"\d{4}-(?:0[1-9]|1[0-2])-(?:0[1-9]|[12]\d|3[01])"
                r"T(?:[01]\d|2[0-3]):[0-5]\d:[0-5]\dZ")


def _parse_ts(value: str):
    """Strict ISO-8601 parse of a single value; naive = UTC. NaT if unparseable.

    Parsed one value at a time on purpose: pd.to_datetime(format="ISO8601") on a
    mixed column can apply one row's offset to a later naive row.
    """
    try:
        ts = pd.Timestamp(datetime.fromisoformat(value)).as_unit("ns")  # outside 1677-2262 raises -> NaT
    except (ValueError, OverflowError):
        return pd.NaT
    return ts.tz_localize("UTC") if ts.tz is None else ts.tz_convert("UTC")


def parse_timestamps(values: pd.Series) -> pd.Series:
    """UTC timestamps for a column of strings (unique index); NaT where unparseable.

    Canonical 'Z' values are parsed in bulk; everything else goes through _parse_ts
    one value at a time, so the result is identical to parsing every value that way.
    """
    fast = values.str.fullmatch(CANONICAL_TS)
    fast_ts = pd.to_datetime(values.where(fast), format="ISO8601", utc=True, errors="coerce")
    slow_ts = pd.to_datetime(values[~fast].map(_parse_ts), utc=True)
    return fast_ts.fillna(slow_ts)


def _normalise(df: pd.DataFrame) -> pd.DataFrame:
    work = df.copy()
    for c in TEXT_COLUMNS:
        work[c] = work[c].fillna("").astype(str).str.strip()
    work["_ts"] = parse_timestamps(work["event_ts"])
    work["ingested_at"] = pd.to_datetime(work["ingested_at"], utc=True)
    return work


def _rules(work: pd.DataFrame, users: pd.DataFrame, cfg: Config):
    """Row-level rules in priority order: (reason_code, failing_mask, describe)."""
    known_users = set(users["user_id"].astype(str).str.strip())
    allowed_types = set(cfg.allowed_event_types)
    latest_allowed = work["ingested_at"] + pd.Timedelta(days=cfg.future_tolerance_days)
    return [
        ("MISSING_EVENT_ID", work["event_id"] == "", lambda w: "event_id is blank"),
        ("MISSING_USER_ID", work["user_id"] == "", lambda w: "user_id is blank"),
        ("MISSING_TIMESTAMP", work["event_ts"] == "", lambda w: "event_ts is blank"),
        ("INVALID_TIMESTAMP", work["_ts"].isna(),
         lambda w: "cannot parse event_ts '" + w["event_ts"] + "'"),
        ("FUTURE_TIMESTAMP", work["_ts"] > latest_allowed,
         lambda w: f"event_ts more than {cfg.future_tolerance_days} day(s) after ingested_at"),
        ("UNKNOWN_USER", ~work["user_id"].isin(known_users),
         lambda w: "user_id '" + w["user_id"] + "' not in users"),
        ("INVALID_EVENT_TYPE", ~work["event_type"].isin(allowed_types),
         lambda w: "event_type '" + w["event_type"] + "' not allowed"),
    ]


def _flag_duplicates(work: pd.DataFrame, reason: pd.Series, detail: pd.Series) -> None:
    """Among rows that passed every rule, keep the earliest copy of each event_id."""
    passed = work[reason.isna()]
    repeated = passed[passed["event_id"].duplicated(keep=False)]  # only these can be duplicates
    if repeated.empty:
        return
    repeated = repeated.sort_values(["ingested_at", "source_file", "line_no"])
    first = repeated.drop_duplicates("event_id", keep="first").set_index("event_id")
    later = repeated[repeated.duplicated("event_id", keep="first")]

    original = first.loc[later["event_id"]]
    original.index = later.index
    same = (later[PAYLOAD_COLUMNS] == original[PAYLOAD_COLUMNS]).all(axis=1)
    reason[later.index] = same.map({True: "DUPLICATE_EXACT", False: "DUPLICATE_CONFLICT"})
    detail[later.index] = ("duplicate of " + original["source_file"]
                           + " line " + original["line_no"].astype(str))


def validate(df: pd.DataFrame, users: pd.DataFrame, cfg: Config):
    """Return (clean, rejects). Every input row ends up in exactly one of them."""
    df = df.reset_index(drop=True)
    work = _normalise(df)
    reason = pd.Series(None, index=work.index, dtype=object)
    detail = pd.Series(None, index=work.index, dtype=object)

    for code, mask, describe in _rules(work, users, cfg):
        hit = mask & reason.isna()
        reason[hit] = code
        detail[hit] = describe(work[hit])

    _flag_duplicates(work, reason, detail)

    clean = work[reason.isna()].copy()
    clean["event_ts"] = clean["_ts"]
    clean.loc[clean["page_url"] == "", "page_url"] = None
    clean = clean.sort_values(["user_id", "event_ts", "event_id"]).reset_index(drop=True)

    rejected = reason.notna()
    rejects = df[rejected].copy()  # raw values, as read from the file
    rejects["reason_code"] = reason[rejected]
    rejects["reason_detail"] = detail[rejected]
    rejects = rejects.sort_values(["source_file", "line_no"]).reset_index(drop=True)

    return clean[CLEAN_COLUMNS], rejects[REJECT_COLUMNS]


Writing pipeline/validate.py


In [17]:
%%writefile tests/test_config.py
import dataclasses

import pytest

from pipeline.config import PROJECT_ROOT, Config


def test_defaults_match_design():
    cfg = Config()
    assert cfg.session_gap_minutes == 30
    assert cfg.future_tolerance_days == 1
    assert cfg.fail_at is None
    assert set(cfg.allowed_event_types) == {"page_view", "click", "add_to_cart", "purchase"}


def test_paths_live_under_project_root():
    cfg = Config()
    for p in (cfg.raw_dir, cfg.users_path, cfg.db_path):
        assert PROJECT_ROOT in p.parents


def test_values_can_be_overridden():
    assert Config(session_gap_minutes=10).session_gap_minutes == 10


@pytest.mark.parametrize("bad", [0, -5])
def test_invalid_gap_rejected(bad):
    with pytest.raises(ValueError):
        Config(session_gap_minutes=bad)


def test_negative_tolerance_rejected():
    with pytest.raises(ValueError):
        Config(future_tolerance_days=-1)


def test_config_is_immutable():
    with pytest.raises(dataclasses.FrozenInstanceError):
        Config().session_gap_minutes = 99


Writing tests/test_config.py


In [18]:
%%writefile tests/test_evidence.py
import shutil

import pytest

from pipeline.config import PROJECT_ROOT, Config
from pipeline.evidence import compare_sessions, fingerprint, time_stages
from pipeline.publish import connect
from pipeline.run import run_pipeline

EXPECTED = PROJECT_ROOT / "data" / "expected" / "expected_sessions.csv"


@pytest.fixture
def env(tmp_path):
    """Sample data loaded once into a private warehouse."""
    shutil.copytree(PROJECT_ROOT / "data" / "raw", tmp_path / "raw")
    shutil.copy(PROJECT_ROOT / "data" / "ref" / "users.csv", tmp_path / "users.csv")
    cfg = Config(raw_dir=tmp_path / "raw", users_path=tmp_path / "users.csv",
                 db_path=tmp_path / "warehouse.duckdb")
    run_pipeline(cfg)
    return cfg


def with_con(cfg, fn):
    con = connect(cfg.db_path)
    try:
        return fn(con)
    finally:
        con.close()


def test_compare_sessions_matches_hand_built_expected(env):
    result = with_con(env, lambda c: compare_sessions(c, EXPECTED))
    assert result == {"expected": 6, "actual": 6, "matched": 6,
                      "missing": 0, "unexpected": 0, "ok": True}


def test_compare_sessions_reports_differences(env):
    def damaged(con):
        con.execute("UPDATE sessions SET event_count = event_count + 1 WHERE user_id = 'u3'")
        return compare_sessions(con, EXPECTED)

    result = with_con(env, damaged)
    assert (result["matched"], result["missing"], result["unexpected"], result["ok"]) == (5, 1, 1, False)


def test_fingerprint_unchanged_by_rerun(env):
    before = with_con(env, fingerprint)
    run_pipeline(env)
    assert with_con(env, fingerprint) == before
    assert before["events_clean"][0] == 13


def test_fingerprint_changes_when_data_changes(env):
    before = with_con(env, fingerprint)
    with_con(env, lambda c: c.execute("UPDATE events_clean SET page_url = '/x' WHERE event_id = 'e001'"))
    after = with_con(env, fingerprint)
    assert after["events_clean"] != before["events_clean"]
    assert after["sessions"] == before["sessions"]


def test_time_stages_returns_each_stage(env):
    df = time_stages(env)
    assert df.stage.tolist() == ["ingest", "validate", "sessionize", "publish"]
    assert (df.seconds > 0).all() and (df.rows_per_second > 0).all()


Writing tests/test_evidence.py


In [19]:
%%writefile tests/test_generate.py
import dataclasses

import pandas as pd
import pytest

from pipeline.config import Config
from pipeline.generate import DEFAULT_RATES, generate_dataset
from pipeline.publish import connect
from pipeline.run import run_pipeline


def read_dir(path):
    frames = [pd.read_csv(p, dtype=str, keep_default_na=False) for p in sorted(path.glob("*.csv"))]
    return pd.concat(frames, ignore_index=True)


def tree_bytes(root):
    return {str(p.relative_to(root)): p.read_bytes() for p in sorted(root.rglob("*")) if p.is_file()}


@pytest.fixture(scope="module")
def data(tmp_path_factory):
    out = tmp_path_factory.mktemp("gen")
    return out, generate_dataset(out, n_events=2000, n_users=100, seed=7, n_files=3)


# --- reproducibility ------------------------------------------------------------

def test_same_seed_gives_identical_files(tmp_path):
    generate_dataset(tmp_path / "a", n_events=500, n_users=40, seed=3, n_files=2)
    generate_dataset(tmp_path / "b", n_events=500, n_users=40, seed=3, n_files=2)
    assert tree_bytes(tmp_path / "a") == tree_bytes(tmp_path / "b")


def test_different_seed_gives_different_data(tmp_path):
    generate_dataset(tmp_path / "a", n_events=500, n_users=40, seed=1, n_files=2)
    generate_dataset(tmp_path / "b", n_events=500, n_users=40, seed=2, n_files=2)
    name = "raw/events_000.csv"
    assert (tmp_path / "a" / name).read_bytes() != (tmp_path / "b" / name).read_bytes()


def test_tiny_dataset_is_rejected(tmp_path):
    with pytest.raises(ValueError):
        generate_dataset(tmp_path, n_events=10)


# --- the files match the manifest -----------------------------------------------

def test_row_counts_match_manifest(data):
    out, m = data
    assert len(read_dir(out / "raw")) == m["rows_written"]["main"]
    assert len(read_dir(out / "late")) == m["rows_written"]["late"] == m["late_events"]


def test_every_reason_code_is_injected(data):
    _, m = data
    assert set(m["injected"]) == set(DEFAULT_RATES)
    assert all(n >= 1 for n in m["injected"].values())
    assert m["expected"]["final"]["rejected_events"] == sum(m["injected"].values())


def test_event_ids_unique_apart_from_injected_duplicates(data):
    out, m = data
    ids = pd.concat([read_dir(out / "raw").event_id, read_dir(out / "late").event_id])
    ids = ids[ids != ""]
    dups = m["injected"]["DUPLICATE_EXACT"] + m["injected"]["DUPLICATE_CONFLICT"]
    assert len(ids) - ids.nunique() == dups


def test_late_events_are_kept_out_of_main_files(data):
    out, _ = data
    assert not set(read_dir(out / "late").event_id) & set(read_dir(out / "raw").event_id)


# --- the known answer -------------------------------------------------------------

def test_truth_sessions_are_consistent(data):
    out, m = data
    truth = pd.read_csv(out / "expected_sessions.csv")
    assert len(truth) == m["sessions"]
    assert truth.event_count.sum() == m["base_events"]
    start = pd.to_datetime(truth.session_start)
    end = pd.to_datetime(truth.session_end)
    assert ((end - start).dt.total_seconds() == truth.duration_seconds).all()


def test_boundary_cases_are_present(data):
    _, m = data
    assert all(n > 0 for n in m["boundary_cases"].values())


# --- end to end on the generated data -----------------------------------------------

def test_pipeline_reproduces_truth_main_then_late(data, tmp_path):
    out, m = data
    base = Config(users_path=out / "ref" / "users.csv", db_path=tmp_path / "w.duckdb")
    main_run = run_pipeline(dataclasses.replace(base, raw_dir=out / "raw"))
    late_run = run_pipeline(dataclasses.replace(base, raw_dir=out / "late"))
    for result, key in ((main_run, "main_run"), (late_run, "late_run")):
        expected = m["expected"][key]
        assert {k: getattr(result, k) for k in expected} == expected

    con = connect(base.db_path)
    got = con.execute("SELECT user_id, session_start, session_end, duration_seconds, event_count "
                      "FROM sessions").df()
    final = {t: con.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0]
             for t in ("events_clean", "sessions", "rejected_events")}
    con.close()
    assert final == m["expected"]["final"]

    fmt = "%Y-%m-%d %H:%M:%S"
    got = got.assign(
        session_start=pd.to_datetime(got.session_start, utc=True).dt.strftime(fmt),
        session_end=pd.to_datetime(got.session_end, utc=True).dt.strftime(fmt),
        duration_seconds=got.duration_seconds.astype(str),
        event_count=got.event_count.astype(str),
    )
    expected = pd.read_csv(out / "expected_sessions.csv", dtype=str, keep_default_na=False)
    key = ["user_id", "session_start"]
    pd.testing.assert_frame_equal(got[list(expected.columns)].sort_values(key).reset_index(drop=True),
                                  expected.sort_values(key).reset_index(drop=True))


Writing tests/test_generate.py


In [20]:
%%writefile tests/test_ingest.py
import pandas as pd
import pytest

from pipeline.config import Config
from pipeline.ingest import (
    EVENT_COLUMNS, LINEAGE_COLUMNS, IngestionError,
    ingest_events, read_events_file, read_users,
)

RUN_TS = pd.Timestamp("2026-10-05 00:00:00", tz="UTC")
HEADER = "event_id,user_id,event_ts,event_type,page_url\n"


def write(tmp_path, name, text):
    p = tmp_path / name
    p.write_text(text, encoding="utf-8")
    return p


# --- happy path on the sample ---------------------------------------------

def test_sample_files_read_completely():
    df = ingest_events(Config().raw_dir, run_id="r1", ingested_at=RUN_TS)
    assert len(df) == 22
    assert list(df.columns) == [*EVENT_COLUMNS, *LINEAGE_COLUMNS]
    assert df.groupby("source_file").size().to_dict() == {"events_a.csv": 11, "events_b.csv": 11}


def test_line_no_points_at_physical_line():
    df = ingest_events(Config().raw_dir, run_id="r1", ingested_at=RUN_TS)
    row = df[(df.source_file == "events_a.csv") & (df.line_no == 10)].iloc[0]
    assert row.event_id == "e030"
    assert df[df.source_file == "events_b.csv"].line_no.min() == 2


def test_lineage_stamped_on_every_row():
    df = ingest_events(Config().raw_dir, run_id="r42", ingested_at=RUN_TS)
    assert (df.run_id == "r42").all()
    assert (df.ingested_at == RUN_TS).all()


def test_naive_ingested_at_is_treated_as_utc():
    df = ingest_events(Config().raw_dir, run_id="r1", ingested_at=pd.Timestamp("2026-10-05"))
    assert df.ingested_at.iloc[0] == RUN_TS


def test_default_ingested_at_is_utc_now():
    df = ingest_events(Config().raw_dir, run_id="r1")
    assert str(df.ingested_at.dt.tz) == "UTC"


# --- raw values are preserved -----------------------------------------------

def test_values_are_kept_as_raw_strings(tmp_path):
    p = write(tmp_path, "f.csv", HEADER + "e1,NA,,click,\n")
    row = read_events_file(p).iloc[0]
    assert row.user_id == "NA"    # not converted to NaN
    assert row.event_ts == ""     # blank stays blank for validation to judge
    assert row.page_url == ""


def test_line_no_survives_blank_lines(tmp_path):
    p = write(tmp_path, "f.csv", HEADER + "e1,u1,2026-10-01T09:00:00Z,click,/a\n\ne2,u1,2026-10-01T09:05:00Z,click,/b\n")
    assert read_events_file(p).line_no.tolist() == [2, 4]


def test_short_row_gives_none_for_missing_fields(tmp_path):
    p = write(tmp_path, "f.csv", HEADER + "e1,u1\n")
    row = read_events_file(p).iloc[0]
    assert row.event_ts is None and row.event_type is None


def test_bom_and_padded_header_are_handled(tmp_path):
    p = write(tmp_path, "f.csv", "\ufeffevent_id, user_id ,event_ts,event_type,page_url\ne1,u1,2026-10-01T09:00:00Z,click,/a\n")
    assert read_events_file(p).iloc[0].user_id == "u1"


def test_header_only_file_gives_zero_rows(tmp_path):
    df = read_events_file(write(tmp_path, "f.csv", HEADER))
    assert len(df) == 0
    assert "line_no" in df.columns


# --- file-level failures fail the run ---------------------------------------

def test_missing_column_fails_with_file_name(tmp_path):
    p = write(tmp_path, "bad.csv", "event_id,user_id,event_type,page_url\ne1,u1,click,/a\n")
    with pytest.raises(IngestionError, match="bad.csv.*event_ts"):
        read_events_file(p)


def test_empty_file_fails(tmp_path):
    with pytest.raises(IngestionError, match="empty.csv"):
        read_events_file(write(tmp_path, "empty.csv", ""))


def test_no_input_files_fails(tmp_path):
    with pytest.raises(IngestionError, match="no .csv files"):
        ingest_events(tmp_path, run_id="r1")


# --- users reference --------------------------------------------------------

def test_users_loaded():
    assert set(read_users(Config().users_path).user_id) == {"u1", "u2", "u3", "u4", "u5"}


def test_missing_users_file_fails(tmp_path):
    with pytest.raises(IngestionError, match="users file not found"):
        read_users(tmp_path / "nope.csv")


Writing tests/test_ingest.py


In [21]:
%%writefile tests/test_notebook.py
import zipfile

import nbformat
import pytest

import build_notebook as bn
from notebook_demo import SECTIONS


@pytest.fixture(scope="module")
def notebook():
    return bn.build_notebook()


def code_sources(nb):
    return [c.source for c in nb.cells if c.cell_type == "code"]


def test_every_source_and_test_file_is_written_verbatim(notebook):
    sources = code_sources(notebook)
    files = [*bn.ROOT.glob("pipeline/*.py"), *bn.ROOT.glob("tests/*.py")]
    assert len(files) > 10
    for path in files:
        name = path.relative_to(bn.ROOT).as_posix()
        assert f"%%writefile {name}\n" + path.read_text(encoding="utf-8") in sources, name


def test_data_docs_and_readme_are_written_too(notebook):
    sources = code_sources(notebook)
    wanted = ["README.md", "pytest.ini", "build_notebook.py", "notebook_demo.py"]
    for pattern in ("data/ref/*.csv", "data/raw/*.csv", "data/expected/*", "docs/*.md"):
        wanted += [p.relative_to(bn.ROOT).as_posix() for p in bn.ROOT.glob(pattern)]
    for name in wanted:
        assert any(s.startswith(f"%%writefile {name}\n") for s in sources), name


def test_notebook_starts_with_readme_and_has_a_section_per_demo(notebook):
    assert notebook.cells[0].cell_type == "markdown"
    assert notebook.cells[0].source == (bn.ROOT / "README.md").read_text(encoding="utf-8")
    headings = [c.source for c in notebook.cells if c.cell_type == "markdown"]
    for title, _ in SECTIONS:
        assert any(h.endswith(title) for h in headings), title
    nbformat.validate(notebook)


def test_zip_holds_sources_and_leaves_out_caches_and_databases(tmp_path):
    out = tmp_path / "project.zip"
    names = bn.make_zip(out)
    with zipfile.ZipFile(out) as archive:
        assert sorted(archive.namelist()) == sorted(names)
    for needed in ("README.md", "pipeline/run.py", "tests/test_run.py", "docs/DECISIONS.md"):
        assert needed in names
    assert not any("__pycache__" in n or n.endswith(".duckdb") for n in names)


Writing tests/test_notebook.py


In [22]:
%%writefile tests/test_publish.py
import pandas as pd
import pytest

from pipeline.config import Config
from pipeline.publish import FAIL_POINTS, InjectedFailure, connect, publish
from pipeline.sessionize import make_session_id
from pipeline.validate import REJECT_COLUMNS

TABLES = ("events_clean", "sessions", "rejected_events")
RUN_TS = pd.Timestamp("2026-10-05", tz="UTC")
CFG = Config()


def ts(s):
    return pd.Timestamp(s, tz="UTC")


def clean(*rows, run_id="r1", source_file="f1.csv"):
    """Validated-style events. rows: (event_id, user_id, 'YYYY-MM-DD HH:MM' UTC, page_url)"""
    df = pd.DataFrame(rows, columns=["event_id", "user_id", "event_ts", "page_url"])
    df["event_ts"] = pd.to_datetime(df["event_ts"], utc=True)
    df["event_type"] = "click"
    df["source_file"] = source_file
    df["ingested_at"] = RUN_TS
    df["run_id"] = run_id
    return df


def rejects(*line_nos, source_file="f1.csv"):
    rows = [{"event_id": f"x{n}", "user_id": "u1", "event_ts": "bad", "event_type": "click",
             "page_url": "/a", "source_file": source_file, "line_no": n,
             "reason_code": "INVALID_TIMESTAMP", "reason_detail": "cannot parse", "run_id": "r1"}
            for n in line_nos]
    return pd.DataFrame(rows, columns=REJECT_COLUMNS)


def batch():
    return clean(("e1", "u1", "2026-10-01 09:00", "/a"),
                 ("e2", "u1", "2026-10-01 09:10", "/b"),
                 ("e3", "u2", "2026-10-01 09:00", "/c"))


def load(con, events, rej=None, files=("f1.csv",), cfg=CFG, run_id="r1"):
    return publish(con, events, rejects() if rej is None else rej, list(files), cfg, run_id)


def snapshot(con, tables=TABLES):
    return {t: con.execute(f"SELECT * FROM {t} ORDER BY ALL").df() for t in tables}


def assert_same(before, after):
    for t in before:
        pd.testing.assert_frame_equal(before[t], after[t])


def count(con, table, where="TRUE"):
    return con.execute(f"SELECT COUNT(*) FROM {table} WHERE {where}").fetchone()[0]


@pytest.fixture
def con():
    c = connect(":memory:")
    yield c
    c.close()


# --- normal loads -------------------------------------------------------------

def test_first_load_writes_events_and_sessions(con):
    result = load(con, batch())
    assert (result.rows_clean, result.rows_already_loaded, result.users_affected) == (3, 0, 2)
    assert result.sessions_written == 2
    assert count(con, "events_clean") == 3 and count(con, "sessions") == 2
    assert count(con, "events_clean", "session_id IS NULL") == 0


def test_rerun_same_batch_changes_nothing(con):
    load(con, batch(), rejects(5))
    before = snapshot(con, ("events_clean", "sessions"))
    result = load(con, batch(), rejects(5), run_id="r2")
    assert (result.rows_clean, result.rows_already_loaded, result.users_affected) == (0, 3, 0)
    assert_same(before, snapshot(con, ("events_clean", "sessions")))
    assert count(con, "rejected_events") == 1


def test_timestamps_round_trip_as_utc(con):
    load(con, batch())
    stored = con.execute("SELECT event_ts FROM events_clean WHERE event_id = 'e1'").df().event_ts.iloc[0]
    assert stored == ts("2026-10-01 09:00")


def test_data_persists_in_database_file(tmp_path):
    path = tmp_path / "warehouse.duckdb"
    c = connect(path)
    load(c, batch())
    c.close()
    c = connect(path)
    assert count(c, "events_clean") == 3 and count(c, "sessions") == 2
    c.close()


def test_empty_batch_is_a_no_op(con):
    load(con, batch())
    before = snapshot(con)
    result = load(con, clean(), files=())
    assert result.rows_clean == 0 and result.users_affected == 0
    assert_same(before, snapshot(con))


# --- late events and existing data --------------------------------------------

def test_late_event_bridges_two_sessions(con):
    load(con, clean(("e1", "u1", "2026-10-01 09:00", "/a"), ("e2", "u1", "2026-10-01 10:00", "/b")))
    assert count(con, "sessions") == 2
    load(con, clean(("e3", "u1", "2026-10-01 09:30", "/c"), run_id="r2"), run_id="r2")
    assert con.execute("SELECT event_count FROM sessions").fetchall() == [(3,)]
    assert count(con, "events_clean", "session_id = (SELECT session_id FROM sessions)") == 3


def test_late_event_before_start_changes_session_id(con):
    load(con, clean(("e1", "u1", "2026-10-01 09:00", "/a")))
    load(con, clean(("e0", "u1", "2026-10-01 08:50", "/z"), run_id="r2"), run_id="r2")
    ids = [r[0] for r in con.execute("SELECT session_id FROM sessions").fetchall()]
    assert ids == [make_session_id("u1", ts("2026-10-01 08:50"))]


def test_only_affected_users_are_rebuilt(con):
    load(con, batch())
    load(con, clean(("e9", "u1", "2026-10-01 12:00", "/x"), run_id="r2"), run_id="r2")
    runs = dict(con.execute("SELECT user_id, max(run_id) FROM sessions GROUP BY user_id").fetchall())
    assert runs == {"u1": "r2", "u2": "r1"}


def test_stored_event_wins_over_conflicting_copy(con):
    load(con, clean(("e1", "u1", "2026-10-01 09:00", "/a")))
    result = load(con, clean(("e1", "u1", "2026-10-01 09:45", "/changed"), run_id="r2"), run_id="r2")
    assert result.rows_already_loaded == 1 and result.rows_clean == 0
    assert con.execute("SELECT page_url FROM events_clean").fetchone()[0] == "/a"


def test_keys_stay_unique_after_many_runs(con):
    for i in range(3):
        load(con, batch(), run_id=f"r{i}")
        load(con, clean(("e9", "u1", "2026-10-01 09:05", "/x")), run_id=f"r{i}")
    distinct = "SELECT COUNT(DISTINCT {k}) FROM {t}"
    assert count(con, "events_clean") == con.execute(distinct.format(k="event_id", t="events_clean")).fetchone()[0]
    assert count(con, "sessions") == con.execute(distinct.format(k="session_id", t="sessions")).fetchone()[0]


# --- rejects ------------------------------------------------------------------

def test_rejects_replaced_per_file_on_rerun(con):
    load(con, batch(), rejects(5, 6))
    load(con, batch(), rejects(5, 6))
    assert count(con, "rejected_events") == 2


def test_rerun_with_fewer_rejects_clears_old_ones(con):
    load(con, batch(), rejects(5, 6))
    load(con, batch(), rejects(6))
    assert con.execute("SELECT line_no FROM rejected_events").fetchall() == [(6,)]


# --- failure and rollback -----------------------------------------------------

@pytest.mark.parametrize("point", FAIL_POINTS)
def test_injected_failure_rolls_back_then_retry_succeeds(con, point):
    load(con, batch(), rejects(5))
    before = snapshot(con)
    late = clean(("e9", "u1", "2026-10-01 09:20", "/x"), run_id="r2")

    with pytest.raises(InjectedFailure):
        load(con, late, rejects(7), cfg=Config(fail_at=point), run_id="r2")
    assert_same(before, snapshot(con))

    load(con, late, rejects(7), run_id="r2")
    assert count(con, "events_clean") == 4


Writing tests/test_publish.py


In [23]:
%%writefile tests/test_reports.py
import dataclasses
import shutil

import pandas as pd
import pytest

from pipeline.config import PROJECT_ROOT, Config
from pipeline.publish import connect
from pipeline.reports import CHECKS, REASON_CODES, REPORTS, main, run_checks, run_report
from pipeline.run import run_pipeline


@pytest.fixture
def env(tmp_path):
    """Sample data loaded once into a private warehouse."""
    shutil.copytree(PROJECT_ROOT / "data" / "raw", tmp_path / "raw")
    shutil.copy(PROJECT_ROOT / "data" / "ref" / "users.csv", tmp_path / "users.csv")
    cfg = Config(raw_dir=tmp_path / "raw", users_path=tmp_path / "users.csv",
                 db_path=tmp_path / "warehouse.duckdb")
    run_pipeline(cfg)
    return cfg


@pytest.fixture
def con(env):
    c = connect(env.db_path)
    yield c
    c.close()


def statuses(con, gap_minutes=30):
    df = run_checks(con, gap_minutes)
    return dict(zip(df["check"], df["status"]))


# --- healthy warehouse ----------------------------------------------------------

def test_all_checks_pass_on_sample(con):
    result = statuses(con)
    assert set(result) == set(CHECKS)
    assert set(result.values()) == {"PASS"}


def test_dq_by_reason_matches_expected_rejects(con):
    df = run_report(con, "dq_by_reason")
    assert set(df.reason_code) == set(REASON_CODES)
    assert (df.row_count == 1).all()
    assert df.pct_of_rows.tolist() == pytest.approx([100 / 22] * 9, abs=0.01)


def test_session_metrics_on_sample(con):
    row = run_report(con, "session_metrics").iloc[0]
    assert (row.sessions, row.users, row.max_duration_s) == (6, 4, 3000)
    assert row.avg_events == pytest.approx(2.17)
    assert row.median_duration_s == pytest.approx(150.0)
    assert row.pct_single_event == pytest.approx(50.0)


def test_sessions_per_day_on_sample(con):
    df = run_report(con, "sessions_per_day")
    assert len(df) == 1
    assert str(df.day.iloc[0]) == "2026-10-01"
    assert (int(df.sessions.iloc[0]), int(df.events.iloc[0])) == (6, 13)


def test_every_report_runs_and_run_history_shows_the_run(con):
    for name in REPORTS:
        assert isinstance(run_report(con, name), pd.DataFrame)
    history = run_report(con, "run_history")
    assert history.status.tolist() == ["SUCCESS"]


# --- each check catches the damage it is meant to catch ---------------------------

@pytest.mark.parametrize("damage, check", [
    ("INSERT INTO events_clean SELECT * FROM events_clean WHERE event_id = 'e001'", "event_id_unique"),
    ("INSERT INTO sessions SELECT * FROM sessions WHERE user_id = 'u3'", "session_id_unique"),
    ("DELETE FROM sessions WHERE user_id = 'u3'", "events_have_session"),
    ("DELETE FROM events_clean WHERE user_id = 'u3'", "sessions_have_events"),
    ("UPDATE sessions SET event_count = event_count + 1 WHERE user_id = 'u3'",
     "session_summary_matches_events"),
    ("UPDATE events_clean SET session_id = (SELECT session_id FROM sessions "
     "WHERE user_id = 'u1' ORDER BY session_start LIMIT 1) WHERE event_id = 'e005'",
     "gap_inside_session_within_threshold"),
    ("UPDATE rejected_events SET reason_code = 'BOGUS' WHERE event_id = 'e030'", "reject_reasons_known"),
    ("INSERT INTO pipeline_runs (run_id, started_at, status) VALUES ('stuck', now(), 'RUNNING')",
     "no_runs_stuck_running"),
    ("UPDATE pipeline_runs SET rows_read = rows_read + 1", "run_counts_reconcile"),
])
def test_checks_catch_damage(con, damage, check):
    con.execute(damage)
    assert statuses(con)[check] == "FAIL"


def test_gap_checks_catch_wrong_threshold(con):
    result = statuses(con, gap_minutes=60)  # data was built with 30; 60 must disagree
    assert result["gap_between_sessions_over_threshold"] == "FAIL"
    assert result["gap_inside_session_within_threshold"] == "PASS"


# --- command line -----------------------------------------------------------------

def test_cli_exit_codes(env, tmp_path):
    assert main(["--db", str(env.db_path)]) == 0
    assert main(["--db", str(tmp_path / "missing.duckdb")]) == 1

    c = connect(env.db_path)
    c.execute("DELETE FROM sessions WHERE user_id = 'u3'")
    c.close()
    assert main(["--db", str(env.db_path)]) == 1


Writing tests/test_reports.py


In [24]:
%%writefile tests/test_run.py
import dataclasses
import json
import shutil

import duckdb
import pandas as pd
import pytest

import pipeline.run as run_mod
from pipeline.config import PROJECT_ROOT, Config
from pipeline.ingest import IngestionError
from pipeline.publish import InjectedFailure
from pipeline.run import ReconciliationError, main, run_pipeline


@pytest.fixture
def env(tmp_path):
    """A private copy of the sample data and an empty warehouse for each test."""
    shutil.copytree(PROJECT_ROOT / "data" / "raw", tmp_path / "raw")
    shutil.copy(PROJECT_ROOT / "data" / "ref" / "users.csv", tmp_path / "users.csv")
    return Config(raw_dir=tmp_path / "raw", users_path=tmp_path / "users.csv",
                  db_path=tmp_path / "warehouse.duckdb")


def query(cfg, sql):
    with duckdb.connect(str(cfg.db_path)) as con:
        return con.execute(sql).df()


def runs(cfg):
    return query(cfg, "SELECT * FROM pipeline_runs ORDER BY started_at, run_id")


def table_count(cfg, table):
    return int(query(cfg, f"SELECT COUNT(*) AS n FROM {table}").n.iloc[0])


def reconciles(row):
    return row.rows_read == row.rows_clean + row.rows_rejected + row.rows_already_loaded


# --- successful runs ----------------------------------------------------------

def test_successful_run_is_logged_with_expected_counts(env):
    result = run_pipeline(env)
    expected = json.loads((PROJECT_ROOT / "data" / "expected" / "expected_run_summary.json").read_text())
    row = runs(env).iloc[0]
    assert result.status == row.status == "SUCCESS"
    for name, value in expected.items():
        assert row[name] == value, name
    assert pd.notna(row.finished_at) and pd.isna(row.error_message)


def test_rerun_is_logged_and_both_runs_reconcile(env):
    run_pipeline(env)
    second = run_pipeline(env)
    assert (second.rows_clean, second.rows_already_loaded) == (0, 13)
    log = runs(env)
    assert list(log.status) == ["SUCCESS", "SUCCESS"]
    assert all(reconciles(r) for r in log.itertuples())
    assert table_count(env, "events_clean") == 13


def test_generated_run_ids_are_unique(env):
    a, b = run_pipeline(env), run_pipeline(env)
    assert a.run_id.startswith("run_") and a.run_id != b.run_id


# --- failures are logged and leave targets untouched -------------------------

def test_injected_failure_logged_and_rolled_back(env):
    with pytest.raises(InjectedFailure):
        run_pipeline(dataclasses.replace(env, fail_at="after_events"))
    row = runs(env).iloc[0]
    assert row.status == "FAILED"
    assert "InjectedFailure" in row.error_message
    assert row.rows_read == 22 and pd.isna(row.rows_clean)
    assert table_count(env, "events_clean") == 0 and table_count(env, "sessions") == 0


def test_retry_after_failure_succeeds(env):
    with pytest.raises(InjectedFailure):
        run_pipeline(dataclasses.replace(env, fail_at="after_sessions"))
    result = run_pipeline(env)
    assert result.rows_clean == 13
    assert list(runs(env).status) == ["FAILED", "SUCCESS"]


def test_ingestion_failure_is_logged(env, tmp_path):
    (tmp_path / "empty").mkdir()
    with pytest.raises(IngestionError):
        run_pipeline(dataclasses.replace(env, raw_dir=tmp_path / "empty"))
    row = runs(env).iloc[0]
    assert row.status == "FAILED" and "no .csv files" in row.error_message
    assert pd.isna(row.rows_read)


def test_missing_users_file_is_logged(env, tmp_path):
    with pytest.raises(IngestionError):
        run_pipeline(dataclasses.replace(env, users_path=tmp_path / "missing.csv"))
    assert runs(env).iloc[0].status == "FAILED"


def test_reconciliation_mismatch_blocks_publish(env, monkeypatch):
    real_validate = run_mod.validate

    def lossy_validate(*args, **kwargs):
        clean, rejects = real_validate(*args, **kwargs)
        return clean.iloc[1:], rejects  # silently lose one row

    monkeypatch.setattr(run_mod, "validate", lossy_validate)
    with pytest.raises(ReconciliationError):
        run_pipeline(env)
    assert runs(env).iloc[0].status == "FAILED"
    assert table_count(env, "events_clean") == 0


def test_duplicate_run_id_is_refused(env):
    run_pipeline(env, run_id="fixed")
    with pytest.raises(ValueError, match="already exists"):
        run_pipeline(env, run_id="fixed")
    assert len(runs(env)) == 1


# --- command line -------------------------------------------------------------

def test_cli_exit_codes(env):
    args = ["--raw-dir", str(env.raw_dir), "--users", str(env.users_path), "--db", str(env.db_path)]
    assert main(args) == 0
    assert main([*args, "--fail-at", "before_commit"]) == 1
    assert list(runs(env).status) == ["SUCCESS", "FAILED"]


Writing tests/test_run.py


In [25]:
%%writefile tests/test_sample_data.py
"""Sanity checks that the hand-built sample and its expected answers agree."""
import csv
import json
from datetime import datetime

from pipeline.config import PROJECT_ROOT

DATA = PROJECT_ROOT / "data"
ALL_REASON_CODES = {
    "MISSING_EVENT_ID", "MISSING_USER_ID", "MISSING_TIMESTAMP", "INVALID_TIMESTAMP",
    "FUTURE_TIMESTAMP", "UNKNOWN_USER", "INVALID_EVENT_TYPE",
    "DUPLICATE_EXACT", "DUPLICATE_CONFLICT",
}


def read_csv(path):
    with open(path, newline="") as f:
        return list(csv.DictReader(f))


def raw_rows():
    """{(file_name, line_no): row} - header is line 1, so data starts at 2."""
    rows = {}
    for path in sorted((DATA / "raw").glob("*.csv")):
        for line_no, row in enumerate(read_csv(path), start=2):
            rows[(path.name, line_no)] = row
    return rows


def summary():
    return json.loads((DATA / "expected" / "expected_run_summary.json").read_text())


def test_summary_reconciles_with_raw_files():
    s = summary()
    assert s["rows_read"] == len(raw_rows())
    assert s["rows_read"] == s["rows_clean"] + s["rows_rejected"] + s["rows_already_loaded"]


def test_every_expected_reject_points_at_a_real_raw_line():
    raw = raw_rows()
    rejects = read_csv(DATA / "expected" / "expected_rejects.csv")
    assert len(rejects) == summary()["rows_rejected"]
    for r in rejects:
        key = (r["source_file"], int(r["line_no"]))
        assert key in raw, f"no raw line for {key}"
        assert raw[key]["event_id"] == r["event_id"]


def test_sample_covers_every_reason_code():
    rejects = read_csv(DATA / "expected" / "expected_rejects.csv")
    assert {r["reason_code"] for r in rejects} == ALL_REASON_CODES


def test_session_event_counts_add_up_to_clean_rows():
    sessions = read_csv(DATA / "expected" / "expected_sessions.csv")
    assert len(sessions) == summary()["sessions_written"]
    assert sum(int(s["event_count"]) for s in sessions) == summary()["rows_clean"]


def test_session_durations_match_start_and_end():
    for s in read_csv(DATA / "expected" / "expected_sessions.csv"):
        start = datetime.fromisoformat(s["session_start"])
        end = datetime.fromisoformat(s["session_end"])
        assert end >= start
        assert int(s["duration_seconds"]) == (end - start).total_seconds()


def test_session_users_exist_and_unknown_user_does_not():
    users = {u["user_id"] for u in read_csv(DATA / "ref" / "users.csv")}
    sessions = read_csv(DATA / "expected" / "expected_sessions.csv")
    assert {s["user_id"] for s in sessions} <= users
    assert "u999" not in users


Writing tests/test_sample_data.py


In [26]:
%%writefile tests/test_sessionize.py
import pandas as pd
import pytest

from pipeline.config import PROJECT_ROOT, Config
from pipeline.ingest import ingest_events, read_users
from pipeline.sessionize import SESSION_COLUMNS, make_session_id, sessionize
from pipeline.validate import validate

RUN_TS = pd.Timestamp("2026-10-05", tz="UTC")


def ev(*rows):
    """rows: (event_id, user_id, 'YYYY-MM-DD HH:MM:SS' in UTC, page_url)"""
    df = pd.DataFrame(rows, columns=["event_id", "user_id", "event_ts", "page_url"])
    df["event_ts"] = pd.to_datetime(df["event_ts"], utc=True)
    df["event_type"] = "click"
    return df


def sess(df, gap=30):
    return sessionize(df, Config(session_gap_minutes=gap), run_id="r1")


# --- the sample matches the hand-built expected answers ------------------------

def test_sample_sessions_match_expected():
    cfg = Config()
    raw = ingest_events(cfg.raw_dir, run_id="r1", ingested_at=RUN_TS)
    clean, _ = validate(raw, read_users(cfg.users_path), cfg)
    _, sessions = sessionize(clean, cfg, run_id="r1")

    expected = pd.read_csv(PROJECT_ROOT / "data" / "expected" / "expected_sessions.csv",
                           dtype=str, keep_default_na=False)
    got = sessions.assign(
        session_start=sessions.session_start.dt.strftime("%Y-%m-%d %H:%M:%S"),
        session_end=sessions.session_end.dt.strftime("%Y-%m-%d %H:%M:%S"),
        duration_seconds=sessions.duration_seconds.astype(str),
        event_count=sessions.event_count.astype(str),
        entry_page=sessions.entry_page.fillna(""),
        exit_page=sessions.exit_page.fillna(""),
    )[list(expected.columns)]

    key = ["user_id", "session_start"]
    pd.testing.assert_frame_equal(got.sort_values(key).reset_index(drop=True),
                                  expected.sort_values(key).reset_index(drop=True))


# --- gap rule and boundaries ---------------------------------------------------

@pytest.mark.parametrize("second_ts, expected_sessions", [
    ("2026-10-01 09:30:00", 1),  # gap exactly 30:00 -> same session
    ("2026-10-01 09:30:01", 2),  # gap 30:01 -> new session
])
def test_gap_boundary(second_ts, expected_sessions):
    _, sessions = sess(ev(("e1", "u1", "2026-10-01 09:00:00", "/a"),
                          ("e2", "u1", second_ts, "/b")))
    assert len(sessions) == expected_sessions


def test_gap_threshold_is_configurable():
    df = ev(("e1", "u1", "2026-10-01 09:00:00", "/a"), ("e2", "u1", "2026-10-01 09:15:00", "/b"))
    assert len(sess(df, gap=30)[1]) == 1
    assert len(sess(df, gap=10)[1]) == 2


def test_session_crossing_midnight_is_not_split():
    _, sessions = sess(ev(("e1", "u1", "2026-10-01 23:50:00", "/a"),
                          ("e2", "u1", "2026-10-02 00:10:00", "/b")))
    assert len(sessions) == 1
    assert sessions.duration_seconds.iloc[0] == 1200


def test_single_event_session_has_zero_duration():
    _, sessions = sess(ev(("e1", "u1", "2026-10-01 09:00:00", "/a")))
    row = sessions.iloc[0]
    assert row.event_count == 1 and row.duration_seconds == 0
    assert row.entry_page == row.exit_page == "/a"


# --- ordering ------------------------------------------------------------------

def test_out_of_order_input_gives_same_sessions():
    df = ev(("e1", "u1", "2026-10-01 09:00:00", "/a"), ("e2", "u1", "2026-10-01 09:10:00", "/b"),
            ("e3", "u1", "2026-10-01 10:00:00", "/c"), ("e4", "u2", "2026-10-01 09:05:00", "/d"))
    _, in_order = sess(df)
    _, shuffled = sess(df.sample(frac=1, random_state=7))
    _, reversed_ = sess(df.iloc[::-1])
    pd.testing.assert_frame_equal(in_order, shuffled)
    pd.testing.assert_frame_equal(in_order, reversed_)


def test_identical_timestamps_ordered_by_event_id():
    events, sessions = sess(ev(("e2", "u1", "2026-10-01 09:00:00", "/b"),
                               ("e1", "u1", "2026-10-01 09:00:00", "/a")))
    assert events.event_id.tolist() == ["e1", "e2"]
    assert sessions.entry_page.iloc[0] == "/a" and sessions.exit_page.iloc[0] == "/b"


def test_users_are_sessionized_independently():
    _, sessions = sess(ev(("e1", "u1", "2026-10-01 09:00:00", "/a"),
                          ("e2", "u2", "2026-10-01 09:05:00", "/b")))
    assert sorted(sessions.user_id) == ["u1", "u2"]


def test_null_entry_page_is_kept_not_skipped():
    _, sessions = sess(ev(("e1", "u1", "2026-10-01 09:00:00", None),
                          ("e2", "u1", "2026-10-01 09:05:00", "/b")))
    assert pd.isna(sessions.entry_page.iloc[0])
    assert sessions.exit_page.iloc[0] == "/b"


# --- session ids and linkage ---------------------------------------------------

def test_session_id_is_deterministic():
    df = ev(("e1", "u1", "2026-10-01 09:00:00", "/a"))
    first, second = sess(df)[1], sess(df)[1]
    sid = first.session_id.iloc[0]
    assert sid == second.session_id.iloc[0]
    assert sid == make_session_id("u1", pd.Timestamp("2026-10-01 09:00", tz="UTC"))
    assert len(sid) == 16


def test_session_id_differs_per_user_with_same_start():
    ts = pd.Timestamp("2026-10-01 09:00", tz="UTC")
    assert make_session_id("u1", ts) != make_session_id("u2", ts)


def test_every_event_links_to_a_session():
    df = ev(("e1", "u1", "2026-10-01 09:00:00", "/a"), ("e2", "u1", "2026-10-01 11:00:00", "/b"),
            ("e3", "u2", "2026-10-01 09:00:00", "/c"))
    events, sessions = sess(df)
    assert events.session_id.notna().all()
    assert set(events.session_id) == set(sessions.session_id)
    assert sessions.event_count.sum() == len(events)


def test_run_id_stamped_on_sessions():
    _, sessions = sess(ev(("e1", "u1", "2026-10-01 09:00:00", "/a")))
    assert (sessions.run_id == "r1").all()


def test_empty_input_gives_empty_outputs():
    events, sessions = sess(ev())
    assert events.empty and sessions.empty
    assert list(sessions.columns) == SESSION_COLUMNS


Writing tests/test_sessionize.py


In [27]:
%%writefile tests/test_timestamps.py
import pandas as pd

import pipeline.validate as validate_mod
from pipeline.config import Config
from pipeline.generate import generate_dataset
from pipeline.validate import parse_timestamps, validate

TRICKY = [
    "2026-10-01T09:00:00Z", "2026-10-01T09:00:00+00:00", "2026-10-02T01:00:00+02:00",
    "2026-10-02 00:20:00", "2026-10-01T09:00:00.123Z", "2026-10-01", "20261001T090000Z",
    "2024-02-29T12:00:00Z", "2026-02-30T00:00:00Z", "2026-13-01T00:00:00Z",
    "2026-10-01T24:00:00Z", "2026-10-01T09:00:60Z", "2026-10-01T09:60:00Z",
    "", "not-a-date", "0000-01-01T00:00:00Z", "9999-12-31T23:59:59Z", "2099-01-01T00:00:00Z",
    "9999-12-31T23:59:59+00:00", "1500-01-01 00:00:00",
]


def per_value(values: pd.Series) -> pd.Series:
    """The reference: every value through the careful one-at-a-time parser."""
    return pd.to_datetime(values.map(validate_mod._parse_ts), utc=True)


def test_matches_per_value_parser_on_tricky_strings():
    values = pd.Series(TRICKY)
    pd.testing.assert_series_equal(parse_timestamps(values), per_value(values), check_names=False)


def test_matches_per_value_parser_on_generated_data(tmp_path):
    generate_dataset(tmp_path, n_events=3000, n_users=100, seed=11, n_files=2)
    frames = [pd.read_csv(p, dtype=str, keep_default_na=False)
              for p in sorted((tmp_path / "raw").glob("*.csv"))]
    values = pd.concat(frames, ignore_index=True)["event_ts"].str.strip()
    assert values.str.fullmatch(validate_mod.CANONICAL_TS).any()   # fast path is exercised
    assert not values.str.fullmatch(validate_mod.CANONICAL_TS).all()  # and so is the slow path
    pd.testing.assert_series_equal(parse_timestamps(values), per_value(values), check_names=False)


def test_canonical_values_use_the_bulk_path(monkeypatch):
    def boom(value):
        raise AssertionError("per-value parser was called for a canonical timestamp")

    monkeypatch.setattr(validate_mod, "_parse_ts", boom)
    parsed = parse_timestamps(pd.Series(["2026-10-01T09:00:00Z", "2026-10-01T09:05:00Z"]))
    assert parsed.tolist() == [pd.Timestamp("2026-10-01 09:00", tz="UTC"),
                               pd.Timestamp("2026-10-01 09:05", tz="UTC")]


def test_naive_value_after_offset_value_stays_utc():
    parsed = parse_timestamps(pd.Series(["2026-10-02T01:00:00+02:00", "2026-10-02 00:20:00"]))
    assert parsed.tolist() == [pd.Timestamp("2026-10-01 23:00", tz="UTC"),
                               pd.Timestamp("2026-10-02 00:20", tz="UTC")]


def test_out_of_range_timestamps_become_nat_not_a_crash():
    # Years outside 1677-2262 cannot be held as a nanosecond timestamp.
    parsed = parse_timestamps(pd.Series(["9999-12-31T23:59:59+00:00", "1500-01-01 00:00:00",
                                         "9999-12-31T23:59:59Z", "2026-10-01T09:00:00Z"]))
    assert parsed.isna().tolist() == [True, True, True, False]


def test_validate_rejects_out_of_range_timestamp_instead_of_failing():
    raw = pd.DataFrame([{"event_id": "e1", "user_id": "u1", "event_ts": "9999-12-31T23:59:59+00:00",
                         "event_type": "click", "page_url": "/a", "source_file": "t.csv",
                         "line_no": 2, "ingested_at": pd.Timestamp("2026-10-05", tz="UTC"),
                         "run_id": "r1"}])
    clean, rejects = validate(raw, pd.DataFrame({"user_id": ["u1"]}), Config())
    assert clean.empty
    assert rejects.reason_code.tolist() == ["INVALID_TIMESTAMP"]


Writing tests/test_timestamps.py


In [28]:
%%writefile tests/test_validate.py
import pandas as pd
import pytest

from pipeline.config import PROJECT_ROOT, Config
from pipeline.ingest import EVENT_COLUMNS, ingest_events, read_users
from pipeline.validate import CLEAN_COLUMNS, REJECT_COLUMNS, validate

RUN_TS = pd.Timestamp("2026-10-05", tz="UTC")
USERS = pd.DataFrame({"user_id": ["u1", "u2"]})
GOOD = {"event_id": "e1", "user_id": "u1", "event_ts": "2026-10-01T09:00:00Z",
        "event_type": "click", "page_url": "/a"}


def make(*overrides):
    """Build an ingested-style frame: one row per override dict, lines 2, 3, ..."""
    rows = [{**GOOD, "source_file": "t.csv", "line_no": i + 2, **o} for i, o in enumerate(overrides)]
    df = pd.DataFrame(rows, columns=[*EVENT_COLUMNS, "source_file", "line_no"])
    df["ingested_at"] = RUN_TS
    df["run_id"] = "r1"
    return df


def run(df):
    return validate(df, USERS, Config())


def reasons(rejects):
    return rejects.reason_code.tolist()


# --- the sample matches the hand-built expected answers ------------------------

def sample():
    cfg = Config()
    raw = ingest_events(cfg.raw_dir, run_id="r1", ingested_at=RUN_TS)
    return validate(raw, read_users(cfg.users_path), cfg)


def test_sample_rejects_match_expected():
    _, rejects = sample()
    expected = pd.read_csv(PROJECT_ROOT / "data" / "expected" / "expected_rejects.csv",
                           dtype=str, keep_default_na=False)
    got = set(zip(rejects.source_file, rejects.line_no.astype(int), rejects.event_id, rejects.reason_code))
    want = set(zip(expected.source_file, expected.line_no.astype(int), expected.event_id, expected.reason_code))
    assert got == want


def test_sample_counts_reconcile():
    clean, rejects = sample()
    assert len(clean) == 13 and len(rejects) == 9
    assert clean.event_id.is_unique


# --- each rule on its own -------------------------------------------------------

@pytest.mark.parametrize("override, code", [
    ({"event_id": ""}, "MISSING_EVENT_ID"),
    ({"event_id": "   "}, "MISSING_EVENT_ID"),
    ({"user_id": ""}, "MISSING_USER_ID"),
    ({"event_ts": ""}, "MISSING_TIMESTAMP"),
    ({"event_ts": None}, "MISSING_TIMESTAMP"),
    ({"event_ts": "not-a-date"}, "INVALID_TIMESTAMP"),
    ({"event_ts": "2026-13-01T00:00:00Z"}, "INVALID_TIMESTAMP"),
    ({"event_ts": "2026-10-07T00:00:00Z"}, "FUTURE_TIMESTAMP"),
    ({"user_id": "u9"}, "UNKNOWN_USER"),
    ({"event_type": "scroll"}, "INVALID_EVENT_TYPE"),
    ({"event_type": "Click"}, "INVALID_EVENT_TYPE"),
])
def test_each_rule(override, code):
    clean, rejects = run(make(override))
    assert clean.empty
    assert reasons(rejects) == [code]


def test_first_failing_rule_wins():
    _, rejects = run(make({"user_id": "", "event_ts": "not-a-date", "event_type": "scroll"}))
    assert reasons(rejects) == ["MISSING_USER_ID"]


def test_future_boundary_exactly_one_day_is_allowed():
    clean, rejects = run(make({"event_ts": "2026-10-06T00:00:00Z"}))
    assert len(clean) == 1 and rejects.empty


def test_timestamps_normalised_to_utc():
    clean, _ = run(make({"event_id": "e1", "event_ts": "2026-10-02T01:00:00+02:00"},
                        {"event_id": "e2", "event_ts": "2026-10-02 00:20:00"}))
    got = dict(zip(clean.event_id, clean.event_ts))
    assert got["e1"] == pd.Timestamp("2026-10-01 23:00", tz="UTC")
    assert got["e2"] == pd.Timestamp("2026-10-02 00:20", tz="UTC")


# --- duplicates ---------------------------------------------------------------

def test_exact_duplicate_keeps_earliest_line():
    clean, rejects = run(make({}, {}))
    assert len(clean) == 1
    assert rejects.line_no.tolist() == [3]
    assert reasons(rejects) == ["DUPLICATE_EXACT"]
    assert rejects.reason_detail.iloc[0] == "duplicate of t.csv line 2"


def test_conflicting_duplicate_flagged_and_first_copy_kept():
    clean, rejects = run(make({}, {"event_ts": "2026-10-01T09:01:00Z"}))
    assert reasons(rejects) == ["DUPLICATE_CONFLICT"]
    assert clean.event_ts.iloc[0] == pd.Timestamp("2026-10-01 09:00", tz="UTC")


def test_same_instant_in_other_format_is_exact_duplicate():
    _, rejects = run(make({}, {"event_ts": "2026-10-01T11:00:00+02:00"}))
    assert reasons(rejects) == ["DUPLICATE_EXACT"]


def test_earliest_copy_decided_by_file_then_line():
    clean, rejects = run(make({"source_file": "b.csv", "line_no": 2},
                              {"source_file": "a.csv", "line_no": 5}))
    assert clean.source_file.tolist() == ["a.csv"]
    assert rejects.source_file.tolist() == ["b.csv"]


def test_invalid_copy_does_not_block_valid_copy():
    clean, rejects = run(make({"event_ts": "not-a-date"}, {}))
    assert len(clean) == 1
    assert reasons(rejects) == ["INVALID_TIMESTAMP"]


# --- output shape -------------------------------------------------------------

def test_clean_values_trimmed_and_blank_page_is_null():
    clean, _ = run(make({"user_id": " u1 ", "event_type": " click ", "page_url": ""}))
    row = clean.iloc[0]
    assert row.user_id == "u1" and row.event_type == "click"
    assert pd.isna(row.page_url)


def test_rejects_keep_raw_values():
    _, rejects = run(make({"event_ts": " not-a-date "}))
    assert rejects.event_ts.iloc[0] == " not-a-date "


def test_empty_input_gives_empty_outputs():
    clean, rejects = run(make())
    assert clean.empty and rejects.empty
    assert list(clean.columns) == CLEAN_COLUMNS
    assert list(rejects.columns) == REJECT_COLUMNS



def test_sample_naive_timestamp_not_shifted_by_neighbouring_offset():
    # Regression: e014 (naive) follows e013 (+02:00) in events_b.csv
    clean, _ = sample()
    got = dict(zip(clean.event_id, clean.event_ts))
    assert got["e013"] == pd.Timestamp("2026-10-01 23:00", tz="UTC")
    assert got["e014"] == pd.Timestamp("2026-10-02 00:20", tz="UTC")


Writing tests/test_validate.py


In [29]:
%%writefile build_notebook.py
"""Builds the submission notebook and the zip from the tested files on disk.

Nothing is copied by hand: every source, test, data and doc file becomes one
%%writefile cell holding that file's exact text.

Usage:  python build_notebook.py [--notebook FILE] [--zip FILE]
"""
import argparse
import zipfile
from pathlib import Path

import nbformat
from nbformat.v4 import new_code_cell, new_markdown_cell, new_notebook

from notebook_demo import SECTIONS, SETUP

ROOT = Path(__file__).resolve().parent

# Files the notebook writes, in order (patterns relative to ROOT).
SOURCE_PATTERNS = ["pytest.ini", "pipeline/*.py", "tests/*.py", "build_notebook.py", "notebook_demo.py",
                   "data/ref/*.csv", "data/raw/*.csv", "data/expected/*", "README.md", "docs/*.md"]
# Files in the zip: the same, plus the evidence outputs and the notebooks.
ZIP_PATTERNS = [*SOURCE_PATTERNS, "data/large/manifest.json", "output/report_*.txt", "*.ipynb"]


def _files(patterns, root: Path) -> list:
    seen, found = set(), []
    for pattern in patterns:
        for path in sorted(root.glob(pattern)):
            if path.is_file() and path not in seen:
                seen.add(path)
                found.append(path)
    return found


def build_notebook(root: Path = ROOT):
    files = _files(SOURCE_PATTERNS, root)
    names = [p.relative_to(root).as_posix() for p in files]
    folders = sorted({Path(n).parent.as_posix() for n in names if Path(n).parent != Path(".")} | {"output"})

    cells = [
        new_markdown_cell((root / "README.md").read_text(encoding="utf-8")),
        new_markdown_cell("## 1. Setup"),
        new_code_cell(SETUP),
        new_markdown_cell("## 2. Project files\n\nEach cell below writes one file, exactly as it was tested."),
        new_code_cell("from pathlib import Path\n\n"
                      f"for folder in {folders!r}:\n    Path(folder).mkdir(parents=True, exist_ok=True)"),
    ]
    for path, name in zip(files, names):
        cells.append(new_code_cell(f"%%writefile {name}\n" + path.read_text(encoding="utf-8")))
    for number, (title, snippets) in enumerate(SECTIONS, start=3):
        cells.append(new_markdown_cell(f"## {number}. {title}"))
        cells.extend(new_code_cell(code) for code in snippets)

    notebook = new_notebook(cells=cells, metadata={
        "kernelspec": {"display_name": "Python 3", "language": "python", "name": "python3"},
        "language_info": {"name": "python"},
    })
    nbformat.validate(notebook)
    return notebook


def make_zip(zip_path, root: Path = ROOT) -> list:
    """Zip sources, tests, data, docs and evidence outputs. Returns the names added."""
    zip_path = Path(zip_path).resolve()
    names = []
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as archive:
        for path in _files(ZIP_PATTERNS, root):
            if path.resolve() == zip_path:
                continue
            name = path.relative_to(root).as_posix()
            archive.write(path, name)
            names.append(name)
    return names


def main(argv=None) -> int:
    parser = argparse.ArgumentParser(description="Build the submission notebook and/or the zip.")
    parser.add_argument("--notebook", type=Path)
    parser.add_argument("--zip", type=Path, dest="zip_path")
    args = parser.parse_args(argv)
    if not (args.notebook or args.zip_path):
        parser.error("give --notebook and/or --zip")
    if args.notebook:
        notebook = build_notebook()
        nbformat.write(notebook, str(args.notebook))
        print(f"wrote {args.notebook}: {len(notebook.cells)} cells")
    if args.zip_path:
        print(f"wrote {args.zip_path}: {len(make_zip(args.zip_path))} files")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


Writing build_notebook.py


In [30]:
%%writefile notebook_demo.py
"""Cells for the submission notebook: setup, then one section per piece of evidence."""

SETUP = '''!pip -q install duckdb pytest nbformat
import sys, platform
import pandas as pd, duckdb, pytest

print("Python :", sys.version.split()[0])
print("Pandas :", pd.__version__)
print("DuckDB :", duckdb.__version__)
print("Pytest :", pytest.__version__)
print("OS     :", platform.platform())'''

SAMPLE_RUN = '''!rm -f output/sample.duckdb
!python -m pipeline.run --db output/sample.duckdb
!python -m pipeline.report --db output/sample.duckdb | tee output/report_sample.txt'''

SAMPLE_COMPARE = '''import json
from pipeline.config import PROJECT_ROOT
from pipeline.evidence import compare_sessions
from pipeline.publish import connect

expected = json.loads((PROJECT_ROOT / "data" / "expected" / "expected_run_summary.json").read_text())
con = connect("output/sample.duckdb")
row = con.execute("SELECT rows_read, rows_clean, rows_rejected, rows_already_loaded, "
                  "sessions_written, users_affected FROM pipeline_runs").df().iloc[0]
print("run summary matches expected:", {k: int(v) for k, v in row.items()} == expected)
print("sessions vs expected_sessions.csv:",
      compare_sessions(con, PROJECT_ROOT / "data" / "expected" / "expected_sessions.csv"))
con.close()'''

FAILURE_RUN = '''!rm -f output/demo.duckdb
!python -m pipeline.run --db output/demo.duckdb --fail-at after_events
!python -m pipeline.run --db output/demo.duckdb'''

FAILURE_LOG = '''from pipeline.publish import connect
from pipeline.reports import run_report

con = connect("output/demo.duckdb")
print(run_report(con, "run_history").to_string(index=False))
print({t: con.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0]
       for t in ("events_clean", "sessions", "rejected_events")})
con.close()'''

LARGE_GENERATE = "!python -m pipeline.generate --events 100000 --users 2000 --seed 42 --out data/large"

LARGE_RUN = '''!rm -f output/large.duckdb
!python -m pipeline.run --raw-dir data/large/raw --users data/large/ref/users.csv --db output/large.duckdb
!python -m pipeline.run --raw-dir data/large/late --users data/large/ref/users.csv --db output/large.duckdb'''

LARGE_CHECK = '''from pipeline.evidence import compare_sessions, fingerprint
from pipeline.publish import connect

con = connect("output/large.duckdb")
print("sessions vs the known answer:", compare_sessions(con, "data/large/expected_sessions.csv"))
first = fingerprint(con)
con.close()
for table, (rows, digest) in first.items():
    print(f"  {table:16} rows={rows:>7}  md5={str(digest)[:12]}")'''

LARGE_REPORT = "!python -m pipeline.report --db output/large.duckdb | tee output/report_large.txt"

RERUN = '''!python -m pipeline.run --raw-dir data/large/raw --users data/large/ref/users.csv --db output/large.duckdb
!python -m pipeline.run --raw-dir data/large/late --users data/large/ref/users.csv --db output/large.duckdb'''

RERUN_CHECK = '''con = connect("output/large.duckdb")
second = fingerprint(con)
con.close()
print("tables unchanged by the rerun:", first == second)'''

BENCH_STAGES = '''import dataclasses
from pipeline.config import PROJECT_ROOT, Config
from pipeline.evidence import time_stages

large = PROJECT_ROOT / "data" / "large"
cfg = dataclasses.replace(Config(), raw_dir=large / "raw", users_path=large / "ref" / "users.csv")
print(time_stages(cfg).to_string(index=False))'''

BENCH_VARIANTS = '''import time
import pandas as pd
import pipeline.validate as val
from pipeline.ingest import ingest_events, read_users

raw = ingest_events(cfg.raw_dir, run_id="bench")
users = read_users(cfg.users_path)


def best_of(fn, repeat=3):
    best = float("inf")
    for _ in range(repeat):
        start = time.perf_counter()
        fn()
        best = min(best, time.perf_counter() - start)
    return best


# Earlier implementations, kept here only for the comparison.
def per_value_parse(values):  # every value through the careful one-at-a-time parser
    return pd.to_datetime(values.map(val._parse_ts), utc=True)


def strptime_parse(values):  # bulk path with an explicit strptime format
    fast = values.str.fullmatch(val.CANONICAL_TS)
    fast_ts = pd.to_datetime(values.where(fast), format="%Y-%m-%dT%H:%M:%SZ", utc=True, errors="coerce")
    slow_ts = pd.to_datetime(values[~fast].map(val._parse_ts), utc=True)
    return fast_ts.fillna(slow_ts)


def sort_everything_flag_duplicates(work, reason, detail):  # sorts every passing row
    passed = work[reason.isna()].sort_values(["ingested_at", "source_file", "line_no"])
    firsts = passed.drop_duplicates("event_id", keep="first").set_index("event_id")
    later = passed[passed.duplicated("event_id", keep="first")]
    if later.empty:
        return
    original = firsts.loc[later["event_id"]]
    original.index = later.index
    same = (later[val.PAYLOAD_COLUMNS] == original[val.PAYLOAD_COLUMNS]).all(axis=1)
    reason[later.index] = same.map({True: "DUPLICATE_EXACT", False: "DUPLICATE_CONFLICT"})
    detail[later.index] = ("duplicate of " + original["source_file"]
                           + " line " + original["line_no"].astype(str))


current_parse, current_flag = val.parse_timestamps, val._flag_duplicates
variants = [
    ("original: per-value parse, sort every row for duplicates", per_value_parse, sort_everything_flag_duplicates),
    ("bulk parse with strptime format", strptime_parse, sort_everything_flag_duplicates),
    ("bulk parse with ISO8601 format", current_parse, sort_everything_flag_duplicates),
    ("+ duplicates: sort only repeated event_ids (current code)", current_parse, current_flag),
]

results, reference = [], None
for label, parse, flag in variants:
    val.parse_timestamps, val._flag_duplicates = parse, flag
    try:
        out = val.validate(raw, users, cfg)
        seconds = best_of(lambda: val.validate(raw, users, cfg))
    finally:
        val.parse_timestamps, val._flag_duplicates = current_parse, current_flag
    if reference is None:
        reference = out
    else:
        pd.testing.assert_frame_equal(reference[0], out[0])
        pd.testing.assert_frame_equal(reference[1], out[1])
    results.append((label, seconds))

table = pd.DataFrame(results, columns=["variant", "validate_s"])
table["vs_original"] = (table.validate_s.iloc[0] / table.validate_s).round(2)
print(table.round(3).to_string(index=False))
print("outputs identical across all variants: True")'''

PACKAGE_ZIP = '''!python build_notebook.py --zip sessionization_project.zip
!unzip -l sessionization_project.zip | tail -n 25'''

PACKAGE_DOWNLOAD = '''try:
    from google.colab import files
    files.download("sessionization_project.zip")
except Exception as exc:  # not running in the Colab front end
    print("download skipped:", exc)'''

# (section title, code cells), in notebook order
SECTIONS = [
    ("Run the tests", ["!python -m pytest -q"]),
    ("Sample data: run, reports, and comparison with the hand-built expected answers",
     [SAMPLE_RUN, SAMPLE_COMPARE]),
    ("Failure and recovery: injected failure, rollback, retry", [FAILURE_RUN, FAILURE_LOG]),
    ("Large dataset: 100,000 events in two batches (main, then late arrivals)",
     [LARGE_GENERATE, LARGE_RUN, LARGE_CHECK, LARGE_REPORT]),
    ("Rerun both batches: nothing may change", [RERUN, RERUN_CHECK]),
    ("Benchmark: where the time goes, and the optimization", [BENCH_STAGES, BENCH_VARIANTS]),
    ("Package the results", [PACKAGE_ZIP, PACKAGE_DOWNLOAD]),
]


Writing notebook_demo.py


In [31]:
%%writefile data/ref/users.csv
user_id,signup_ts,country
u1,2026-01-15T10:00:00Z,US
u2,2026-02-20T11:30:00Z,DE
u3,2026-03-05T09:15:00Z,IN
u4,2026-04-10T14:45:00Z,GB
u5,2026-05-01T08:00:00Z,US


Writing data/ref/users.csv


In [32]:
%%writefile data/raw/events_a.csv
event_id,user_id,event_ts,event_type,page_url
e001,u1,2026-10-01T09:00:00Z,page_view,/home
e003,u1,2026-10-01T09:20:00Z,click,/product/1
e002,u1,2026-10-01T09:10:00Z,page_view,/product/1
e004,u1,2026-10-01T09:50:00Z,add_to_cart,/cart
e005,u1,2026-10-01T10:20:01Z,purchase,/checkout
e020,u3,2026-10-01T12:00:00Z,page_view,/home
e021,u3,2026-10-01T12:05:00Z,click,/product/9
e020,u3,2026-10-01T12:00:00Z,page_view,/home
e030,u4,,page_view,/home
e031,u4,not-a-date,page_view,/home
,u4,2026-10-01T08:00:00Z,page_view,/home


Writing data/raw/events_a.csv


In [33]:
%%writefile data/raw/events_b.csv
event_id,user_id,event_ts,event_type,page_url
e010,u2,2026-10-01T23:50:00Z,page_view,/home
e012,u2,2026-10-02T00:05:00Z,click,/product/3
e011,u2,2026-10-02T00:05:00Z,page_view,/product/2
e013,u2,2026-10-02T01:00:00+02:00,page_view,/search
e014,u2,2026-10-02 00:20:00,click,/cart
e021,u3,2026-10-01T12:06:00Z,click,/product/9
e032,u4,2099-01-01T00:00:00Z,page_view,/home
e033,u4,2026-10-01T08:00:00Z,scroll,/home
e034,u4,2026-10-01T08:10:00Z,page_view,
e035,,2026-10-01T08:00:00Z,page_view,/home
e036,u999,2026-10-01T08:00:00Z,page_view,/home


Writing data/raw/events_b.csv


In [34]:
%%writefile data/expected/expected_rejects.csv
source_file,line_no,event_id,reason_code
events_a.csv,9,e020,DUPLICATE_EXACT
events_a.csv,10,e030,MISSING_TIMESTAMP
events_a.csv,11,e031,INVALID_TIMESTAMP
events_a.csv,12,,MISSING_EVENT_ID
events_b.csv,7,e021,DUPLICATE_CONFLICT
events_b.csv,8,e032,FUTURE_TIMESTAMP
events_b.csv,9,e033,INVALID_EVENT_TYPE
events_b.csv,11,e035,MISSING_USER_ID
events_b.csv,12,e036,UNKNOWN_USER


Writing data/expected/expected_rejects.csv


In [35]:
%%writefile data/expected/expected_run_summary.json
{
  "rows_read": 22,
  "rows_clean": 13,
  "rows_rejected": 9,
  "rows_already_loaded": 0,
  "sessions_written": 6,
  "users_affected": 4
}


Writing data/expected/expected_run_summary.json


In [36]:
%%writefile data/expected/expected_sessions.csv
user_id,session_start,session_end,duration_seconds,event_count,entry_page,exit_page
u1,2026-10-01 09:00:00,2026-10-01 09:50:00,3000,4,/home,/cart
u1,2026-10-01 10:20:01,2026-10-01 10:20:01,0,1,/checkout,/checkout
u2,2026-10-01 23:00:00,2026-10-01 23:00:00,0,1,/search,/search
u2,2026-10-01 23:50:00,2026-10-02 00:20:00,1800,4,/home,/cart
u3,2026-10-01 12:00:00,2026-10-01 12:05:00,300,2,/home,/product/9
u4,2026-10-01 08:10:00,2026-10-01 08:10:00,0,1,,


Writing data/expected/expected_sessions.csv


In [37]:
%%writefile README.md
# Clickstream Sessionization Pipeline

Turns raw clickstream events into per-user sessions using event-time gaps. It handles
out-of-order events, duplicates, missing timestamps and boundary cases; records every run;
rejects bad rows with a reason instead of dropping them silently; and can be re-run safely.

Stack: Python, pandas, DuckDB (warehouse), pytest. Runs in Google Colab or locally.

## Quick start

**Google Colab (documented cloud environment)**

1. Open colab.research.google.com, then File > Upload notebook, and pick `sessionization_submission.ipynb`.
2. Runtime > Run all. The notebook writes the source files and tests, runs pytest, runs the
   pipeline on the sample, then on a generated 100k dataset, and prints the reports.
3. Colab erases its files when the runtime resets. Download the zip from the last cell to keep outputs.

**Locally**

pip install pandas numpy duckdb pytest nbformat
python -m pytest -q
python -m pipeline.run
python -m pipeline.report
python -m pipeline.generate --events 100000 --seed 42
python -m pipeline.run --raw-dir data/large/raw --users data/large/ref/users.csv --db output/large.duckdb
python -m pipeline.run --raw-dir data/large/late --users data/large/ref/users.csv --db output/large.duckdb
python -m pipeline.report --db output/large.duckdb

`pipeline.run` exits with code 0 on success and 1 on failure; `--fail-at` injects a failure
(after_sessions, after_events or before_commit) to demonstrate rollback.

## Layout

    pipeline/
      config.py      settings: 30-minute gap, 1-day future tolerance, paths, fault flag
      ingest.py      read CSVs as raw strings, add lineage columns
      validate.py    9 reason codes, duplicate handling, UTC normalisation
      sessionize.py  gap rule, deterministic session_id
      publish.py     DuckDB tables, one transaction, late-event rebuild
      run_log.py     pipeline_runs table
      run.py         run_pipeline() and the command line
      reports.py     10 reconciliation checks, 7 reports (report.py is the CLI shim)
      synth.py       synthetic events with a known answer
      generate.py    seeded dataset generator
      evidence.py    compare / fingerprint / timing helpers
    tests/           pytest suites (one per module)
    data/            hand-built sample (ref, raw) and its expected answers
    docs/            DESIGN.md (original design), DECISIONS.md (what changed and why)

## Data model

Source (CSV): `raw_events` (event_id, user_id, event_ts, event_type, page_url) and
`users` (user_id, signup_ts, country). Each row also gets source_file, line_no (physical
line, header = 1), ingested_at and run_id.

Target (DuckDB):

| Table | Contents |
|---|---|
| events_clean | one row per valid event, with its session_id |
| sessions | session_id, user_id, start, end, duration_seconds, event_count, entry_page, exit_page |
| rejected_events | the raw row, reason_code, reason_detail, source_file, line_no, run_id |
| pipeline_runs | run_id, status (RUNNING/SUCCESS/FAILED), start/finish, row counts, error_message |

## Rules

Validation, in this order; each rejected row carries the first rule it fails:

| Reason code | Rule |
|---|---|
| MISSING_EVENT_ID / MISSING_USER_ID | blank |
| MISSING_TIMESTAMP | blank event_ts (rejected, not filled with ingestion time: a wrong time would silently corrupt session gaps) |
| INVALID_TIMESTAMP | not ISO-8601, or outside years 1677-2262 |
| FUTURE_TIMESTAMP | more than 1 day after ingestion |
| UNKNOWN_USER | user_id not in users |
| INVALID_EVENT_TYPE | not page_view, click, add_to_cart or purchase (case-sensitive) |
| DUPLICATE_EXACT / DUPLICATE_CONFLICT | same event_id; the earliest copy (ingested_at, source_file, line_no) is kept; CONFLICT means the payload differs |

Timestamps are converted to UTC; a timestamp with no timezone is treated as UTC.

Sessionization:

- Events are ordered per user by (event_ts, event_id); the event_id breaks ties between identical timestamps.
- A new session starts at a user's first event, or when the gap to the previous event is **more than** 30 minutes. A gap of **exactly** 30:00 stays in the same session; 30:01 starts a new one.
- Sessions are not split at midnight or at batch boundaries. A single-event session has duration 0.
- session_id is the first 16 hex characters of sha256(user_id + "|" + session_start in ISO format), so a rerun produces the same ids.

## Reliability

- **One transaction per publish.** Any failure rolls back every change; the run is then recorded as FAILED with the error, and the error is raised again. The run log is written outside that transaction so the FAILED row survives.
- **Safe reruns.** An event_id that is already stored is kept and counted as rows_already_loaded (not a reject). Rejects are replaced per source file. Rerunning the same input leaves all tables unchanged.
- **Late events.** When a later batch brings events for a user, that user's sessions are rebuilt from all their stored and new events and replaced in the same transaction. A late event can join two sessions or move a session's start.
- **Reconciliation.** rows_read = rows_clean + rows_rejected + rows_already_loaded is checked before publishing and again in the reports.

## Data quality and reports

`python -m pipeline.report` prints 10 SQL checks (each counts violations; 0 = PASS) and 7 reports:
session metrics, sessions per day, top entry pages, events by type, rejects by reason, rejects by file, and run history.
The checks include an independent proof of the gap rule: inside a session no gap exceeds 30:00,
and between two sessions of one user the gap is always more than 30:00.

Sample queries:

    SELECT reason_code, COUNT(*) AS n FROM rejected_events GROUP BY 1 ORDER BY 2 DESC;
    SELECT user_id, session_start, duration_seconds, event_count
    FROM sessions ORDER BY duration_seconds DESC LIMIT 5;

## Testing evidence

| Test file | Tests | What it proves |
|---|---|---|
| test_config | 7 | defaults and validation of settings |
| test_sample_data | 6 | the hand-built sample and its expected answers agree |
| test_ingest | 15 | raw values kept, line numbers, file-level failures |
| test_validate | 25 | every reason code, rule order, duplicates, UTC handling |
| test_sessionize | 15 | the 30:00 / 30:01 boundary, midnight, ties, out-of-order input, session ids |
| test_publish | 15 | idempotent rerun, late events, rollback at each failure point |
| test_run | 10 | run log, FAILED then successful retry, reconciliation guard, exit codes |
| test_generate | 10 | same seed gives identical files; pipeline reproduces the generator's known answer |
| test_reports | 16 | all checks pass on healthy data; each check fails when the matching damage is injected |
| test_evidence | 5 | comparison and fingerprint helpers |
| test_timestamps | 6 | bulk timestamp parser equals the careful per-value parser; out-of-range years rejected |

End-to-end results (Colab, seed 42):

- **Sample**: 22 rows -> 13 clean, 9 rejected (all 9 reason codes), 6 sessions; matches the hand-built expected files.
- **100,000 events** in two batches (main batch of 101,300 rows including 3,300 injected bad rows, then a late batch of 2,000 events): 100,000 clean events, 3,300 rejects, **25,079 sessions, identical to the generator's known answer** (0 missing, 0 unexpected). The late batch merged 396 sessions that had been split only because those events were missing.
- All 10 reconciliation checks PASS.
- **Rerun** of both batches: 0 new rows (98,000 and 2,000 already loaded); a fingerprint (row count plus MD5 of content) of all three tables is unchanged.

## Performance

Generated 100k events, Colab: main batch about 4.5-5.8 s, late batch about 1.3-1.8 s (run-to-run variation on Colab is around 30%).
The stage profile showed validation was the slowest stage (about 2.0 s of 4.5 s). One optimization with two parts,
each measured on the real 101,300 rows, best of 3, outputs asserted identical:

| Variant | validate() seconds | Speedup |
|---|---|---|
| original: per-value timestamp parse, sort every row for duplicates | 2.276 | 1.00x |
| bulk parse of canonical timestamps (strptime format) | 2.021 | 1.13x |
| bulk parse using pandas' ISO8601 parser | 1.855 | 1.23x |
| plus sort only repeated event_ids when flagging duplicates | 1.698 | 1.34x |

Only the stage speedup is claimed; whole-run timings vary by more than the gain.

## Assumptions

- All times are UTC; one CSV row is one event; processing is batch, not streaming.
- The users file is complete for the period processed.
- A stored event wins over a later copy with the same event_id.

## Known limitations and future work

- When late events arrive, all sessions of the affected users are rebuilt (2,000 late events rewrote 15,988 sessions). Correct, and fast enough here, but it scales with the affected users' history.
- Session ids are derived from the session start, so a late event that moves a session's start changes its id.
- validate sorts the clean rows and sessionize sorts them again; removing one sort would save about 0.6 s per 100k rows but changes the order validate returns rows in.
- DuckDB allows one writer at a time; there is no concurrent-run protection beyond that.
- Colab storage is temporary; download the zip to keep results.


Writing README.md


In [38]:
%%writefile docs/DECISIONS.md
# Decisions made during the build

Where the build differs from, or adds to, docs/DESIGN.md. Each row names the test that shows it.

| Decision | Why | Evidence |
|---|---|---|
| Read CSVs with the csv module, not pandas | pandas skips blank lines (wrong line_no) and turns strings like "NA" into NaN | test_ingest: blank lines, "NA" user |
| Parse timestamps one value at a time, with a bulk path only for the canonical `YYYY-MM-DDTHH:MM:SSZ` form | pd.to_datetime on a mixed column applied one row's +02:00 offset to a later naive row (event e014 came out 2 hours early) | test_timestamps_normalised_to_utc, test_naive_value_after_offset_value_stays_utc |
| Years outside 1677-2262 are rejected as INVALID_TIMESTAMP | such a value crashed the whole run instead of rejecting one row | test_out_of_range_timestamps_become_nat_not_a_crash, test_validate_rejects_out_of_range_timestamp_instead_of_failing |
| No PRIMARY KEY constraints on target tables | DuckDB checks unique keys eagerly, so delete + re-insert of the same key in one transaction can fail; uniqueness comes from dedup, whole-user replace and the event_id_unique / session_id_unique checks | test_keys_stay_unique_after_many_runs, test_checks_catch_damage |
| pipeline_runs is written outside the data transaction | otherwise the FAILED row would be rolled back with the data | test_injected_failure_logged_and_rolled_back |
| A stored event wins over a re-sent copy, and is not a reject | makes reruns idempotent; keeps rows_read reconciling | test_stored_event_wins_over_conflicting_copy |
| Rejects are replaced per source file | a rerun with fewer problems clears old rejects | test_rerun_with_fewer_rejects_clears_old_ones |
| event_type is matched case-sensitively | keeps the allowed list strict | test_each_rule ("Click") |
| Missing timestamps are rejected, not filled with ingestion time | a wrong time would silently change session gaps | test_each_rule (MISSING_TIMESTAMP) |
| entry/exit page taken from the real first/last event | groupby first/last skips nulls and would return the second page | test_null_entry_page_is_kept_not_skipped |
| session_id depends on the session start | deterministic ids on rerun; changes only when a late event moves the start | test_session_id_is_deterministic, test_late_event_before_start_changes_session_id |
| Duplicates are found among rows that passed every rule, sorting only repeated event_ids | an invalid first copy must not block a valid later copy; avoids sorting every row | test_invalid_copy_does_not_block_valid_copy; 1.34x measurement in README |
| Redundant sort of clean rows left in validate | removing it changes validate's output order; not needed for correctness | listed in README limitations |


Writing docs/DECISIONS.md


In [39]:
%%writefile docs/DESIGN.md
# Sessionization Pipeline - Design

## Source schemas (CSV)
raw_events (data/raw/*.csv):
| column     | type             | notes                                         |
|------------|------------------|-----------------------------------------------|
| event_id   | string           | business key, expected unique                 |
| user_id    | string           | must exist in users                           |
| event_ts   | string->timestamp| ISO-8601, normalised to UTC; naive = UTC      |
| event_type | string           | page_view, click, add_to_cart, purchase       |
| page_url   | string           | nullable                                      |

users (data/ref/users.csv): user_id (key), signup_ts, country

Lineage columns added on ingest: source_file, line_no, ingested_at, run_id.

## Validation rules (applied in order, first failure wins)
| reason_code        | rule                                              |
|--------------------|---------------------------------------------------|
| MISSING_EVENT_ID   | event_id null/blank                               |
| MISSING_USER_ID    | user_id null/blank                                |
| MISSING_TIMESTAMP  | event_ts null/blank                               |
| INVALID_TIMESTAMP  | event_ts unparseable                              |
| FUTURE_TIMESTAMP   | event_ts > ingested_at + 1 day                    |
| UNKNOWN_USER       | user_id not in users                              |
| INVALID_EVENT_TYPE | event_type not in allowed list                    |
| DUPLICATE_EXACT    | same event_id + same payload; keep earliest copy  |
| DUPLICATE_CONFLICT | same event_id, different payload; keep earliest   |

"Earliest copy" = ordered by (ingested_at, source_file, line_no).
Missing timestamps are rejected, not back-filled with ingestion time,
because a wrong time would silently corrupt session gaps.

## Sessionization rules
- Partition by user_id, order by (event_ts, event_id) - event_id breaks ties.
- New session when first event OR gap > 30 minutes.
- Gap == 30 minutes exactly stays in the SAME session.
- No split at midnight or batch boundaries.
- Single-event session => duration 0.
- session_id = first 16 hex chars of sha256(user_id + "|" + session_start ISO).

## Target tables (DuckDB)
- events_clean    (PK event_id): user_id, event_ts, event_type, page_url,
                   session_id, source_file, ingested_at, run_id
- sessions        (PK session_id): user_id, session_start, session_end,
                   duration_seconds, event_count, entry_page, exit_page, run_id
- rejected_events (key source_file, line_no): raw cols, reason_code,
                   reason_detail, run_id
- pipeline_runs   (PK run_id): started_at, finished_at, status
                   (RUNNING/SUCCESS/FAILED), rows_read, rows_clean,
                   rows_rejected, rows_already_loaded, sessions_written,
                   users_affected, error_message

## Late events and rerun safety
- Affected users = users with new clean events in this batch.
- Their sessions are rebuilt from ALL their events (stored + new) and
  replaced in ONE transaction.
- event_id already in events_clean => existing row wins; counted as
  rows_already_loaded (not a reject).
- Rerun of the same file => same final state (idempotent).
- Any failure => rollback, run marked FAILED with error_message.
- Reconciliation: rows_read = rows_clean + rows_rejected + rows_already_loaded

## Assumptions
- All times UTC. One CSV row = one event.
- users reference data is complete for the processed period.
- Batch processing, not streaming.


Writing docs/DESIGN.md


## 3. Run the tests

In [40]:
!python -m pytest -q

........................................................................ [ 53%]
..............................................................           [100%]
134 passed in 22.52s


## 4. Sample data: run, reports, and comparison with the hand-built expected answers

In [41]:
!rm -f output/sample.duckdb
!python -m pipeline.run --db output/sample.duckdb
!python -m pipeline.report --db output/sample.duckdb | tee output/report_sample.txt

Run SUCCESS: RunResult(run_id='run_20261101T130940_9cf8a4', status='SUCCESS', rows_read=22, rows_clean=13, rows_rejected=9, rows_already_loaded=0, sessions_written=6, users_affected=4)
== reconciliation checks ==
                              check  violations status
               run_counts_reconcile           0   PASS
              no_runs_stuck_running           0   PASS
                    event_id_unique           0   PASS
                  session_id_unique           0   PASS
                events_have_session           0   PASS
               sessions_have_events           0   PASS
     session_summary_matches_events           0   PASS
gap_inside_session_within_threshold           0   PASS
gap_between_sessions_over_threshold           0   PASS
               reject_reasons_known           0   PASS

== session_metrics ==
 sessions  users  avg_events  avg_duration_s  median_duration_s  p95_duration_s  max_duration_s  pct_single_event
        6      4        2.17           850.0 

In [42]:
import json
from pipeline.config import PROJECT_ROOT
from pipeline.evidence import compare_sessions
from pipeline.publish import connect

expected = json.loads((PROJECT_ROOT / "data" / "expected" / "expected_run_summary.json").read_text())
con = connect("output/sample.duckdb")
row = con.execute("SELECT rows_read, rows_clean, rows_rejected, rows_already_loaded, "
                  "sessions_written, users_affected FROM pipeline_runs").df().iloc[0]
print("run summary matches expected:", {k: int(v) for k, v in row.items()} == expected)
print("sessions vs expected_sessions.csv:",
      compare_sessions(con, PROJECT_ROOT / "data" / "expected" / "expected_sessions.csv"))
con.close()

run summary matches expected: True
sessions vs expected_sessions.csv: {'expected': 6, 'actual': 6, 'matched': 6, 'missing': 0, 'unexpected': 0, 'ok': True}


## 5. Failure and recovery: injected failure, rollback, retry

In [43]:
!rm -f output/demo.duckdb
!python -m pipeline.run --db output/demo.duckdb --fail-at after_events
!python -m pipeline.run --db output/demo.duckdb

Run FAILED: InjectedFailure: injected failure at 'after_events'
Run SUCCESS: RunResult(run_id='run_20261101T130942_0ec7f4', status='SUCCESS', rows_read=22, rows_clean=13, rows_rejected=9, rows_already_loaded=0, sessions_written=6, users_affected=4)


In [44]:
from pipeline.publish import connect
from pipeline.reports import run_report

con = connect("output/demo.duckdb")
print(run_report(con, "run_history").to_string(index=False))
print({t: con.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0]
       for t in ("events_clean", "sessions", "rejected_events")})
con.close()

                    run_id  status  rows_read  rows_clean  rows_rejected  rows_already_loaded  sessions_written  users_affected  seconds                                       error_message
run_20261101T130942_792909  FAILED         22        <NA>           <NA>                 <NA>              <NA>            <NA>     0.08 InjectedFailure: injected failure at 'after_events'
run_20261101T130942_0ec7f4 SUCCESS         22          13              9                    0                 6               4     0.14                                                None
{'events_clean': 13, 'sessions': 6, 'rejected_events': 9}


## 6. Large dataset: 100,000 events in two batches (main, then late arrivals)

In [45]:
!python -m pipeline.generate --events 100000 --users 2000 --seed 42 --out data/large

{
  "base_events": 100000,
  "boundary_cases": {
    "identical_timestamp_pairs": 2142,
    "new_session_gap_exactly_1801s": 1113,
    "same_session_gap_exactly_1800s": 1533
  },
  "expected": {
    "final": {
      "events_clean": 100000,
      "rejected_events": 3300,
      "sessions": 25079
    },
    "late_run": {
      "rows_already_loaded": 0,
      "rows_clean": 2000,
      "rows_read": 2000,
      "rows_rejected": 0
    },
    "main_run": {
      "rows_already_loaded": 0,
      "rows_clean": 98000,
      "rows_read": 101300,
      "rows_rejected": 3300
    }
  },
  "injected": {
    "DUPLICATE_CONFLICT": 300,
    "DUPLICATE_EXACT": 1000,
    "FUTURE_TIMESTAMP": 200,
    "INVALID_EVENT_TYPE": 300,
    "INVALID_TIMESTAMP": 500,
    "MISSING_EVENT_ID": 100,
    "MISSING_TIMESTAMP": 500,
    "MISSING_USER_ID": 100,
    "UNKNOWN_USER": 300
  },
  "late_events": 2000,
  "n_files": 5,
  "n_users": 2000,
  "rows_written": {
    "late": 2000,
    "main": 101300
  },
  "seed": 42,
  "ses

In [46]:
!rm -f output/large.duckdb
!python -m pipeline.run --raw-dir data/large/raw --users data/large/ref/users.csv --db output/large.duckdb
!python -m pipeline.run --raw-dir data/large/late --users data/large/ref/users.csv --db output/large.duckdb

Run SUCCESS: RunResult(run_id='run_20261101T130949_5633a1', status='SUCCESS', rows_read=101300, rows_clean=98000, rows_rejected=3300, rows_already_loaded=0, sessions_written=25475, users_affected=2000)
Run SUCCESS: RunResult(run_id='run_20261101T130955_92bd3f', status='SUCCESS', rows_read=2000, rows_clean=2000, rows_rejected=0, rows_already_loaded=0, sessions_written=15988, users_affected=1221)


In [47]:
from pipeline.evidence import compare_sessions, fingerprint
from pipeline.publish import connect

con = connect("output/large.duckdb")
print("sessions vs the known answer:", compare_sessions(con, "data/large/expected_sessions.csv"))
first = fingerprint(con)
con.close()
for table, (rows, digest) in first.items():
    print(f"  {table:16} rows={rows:>7}  md5={str(digest)[:12]}")

sessions vs the known answer: {'expected': 25079, 'actual': 25079, 'matched': 25079, 'missing': 0, 'unexpected': 0, 'ok': True}
  events_clean     rows= 100000  md5=e749ef5b8798
  sessions         rows=  25079  md5=be9eb51f0acc
  rejected_events  rows=   3300  md5=9abc28c58ab3


In [48]:
!python -m pipeline.report --db output/large.duckdb | tee output/report_large.txt

== reconciliation checks ==
                              check  violations status
               run_counts_reconcile           0   PASS
              no_runs_stuck_running           0   PASS
                    event_id_unique           0   PASS
                  session_id_unique           0   PASS
                events_have_session           0   PASS
               sessions_have_events           0   PASS
     session_summary_matches_events           0   PASS
gap_inside_session_within_threshold           0   PASS
gap_between_sessions_over_threshold           0   PASS
               reject_reasons_known           0   PASS

== session_metrics ==
 sessions  users  avg_events  avg_duration_s  median_duration_s  p95_duration_s  max_duration_s  pct_single_event
    25079   2000        3.99          2577.7             1584.0          8776.8           30285              25.2

== sessions_per_day ==
       day  sessions  events
2026-01-01      3926   15423
2026-01-02      9356   37046
2026-

## 7. Rerun both batches: nothing may change

In [49]:
!python -m pipeline.run --raw-dir data/large/raw --users data/large/ref/users.csv --db output/large.duckdb
!python -m pipeline.run --raw-dir data/large/late --users data/large/ref/users.csv --db output/large.duckdb

Run SUCCESS: RunResult(run_id='run_20261101T130958_00cf86', status='SUCCESS', rows_read=101300, rows_clean=0, rows_rejected=3300, rows_already_loaded=98000, sessions_written=0, users_affected=0)
Run SUCCESS: RunResult(run_id='run_20261101T131001_dc659f', status='SUCCESS', rows_read=2000, rows_clean=0, rows_rejected=0, rows_already_loaded=2000, sessions_written=0, users_affected=0)


In [50]:
con = connect("output/large.duckdb")
second = fingerprint(con)
con.close()
print("tables unchanged by the rerun:", first == second)

tables unchanged by the rerun: True


## 8. Benchmark: where the time goes, and the optimization

In [51]:
import dataclasses
from pipeline.config import PROJECT_ROOT, Config
from pipeline.evidence import time_stages

large = PROJECT_ROOT / "data" / "large"
cfg = dataclasses.replace(Config(), raw_dir=large / "raw", users_path=large / "ref" / "users.csv")
print(time_stages(cfg).to_string(index=False))

     stage  seconds  rows_per_second
    ingest    0.483         209594.0
  validate    1.701          59562.0
sessionize    0.927         109269.0
   publish    1.534          66027.0


In [52]:
import time
import pandas as pd
import pipeline.validate as val
from pipeline.ingest import ingest_events, read_users

raw = ingest_events(cfg.raw_dir, run_id="bench")
users = read_users(cfg.users_path)


def best_of(fn, repeat=3):
    best = float("inf")
    for _ in range(repeat):
        start = time.perf_counter()
        fn()
        best = min(best, time.perf_counter() - start)
    return best


# Earlier implementations, kept here only for the comparison.
def per_value_parse(values):  # every value through the careful one-at-a-time parser
    return pd.to_datetime(values.map(val._parse_ts), utc=True)


def strptime_parse(values):  # bulk path with an explicit strptime format
    fast = values.str.fullmatch(val.CANONICAL_TS)
    fast_ts = pd.to_datetime(values.where(fast), format="%Y-%m-%dT%H:%M:%SZ", utc=True, errors="coerce")
    slow_ts = pd.to_datetime(values[~fast].map(val._parse_ts), utc=True)
    return fast_ts.fillna(slow_ts)


def sort_everything_flag_duplicates(work, reason, detail):  # sorts every passing row
    passed = work[reason.isna()].sort_values(["ingested_at", "source_file", "line_no"])
    firsts = passed.drop_duplicates("event_id", keep="first").set_index("event_id")
    later = passed[passed.duplicated("event_id", keep="first")]
    if later.empty:
        return
    original = firsts.loc[later["event_id"]]
    original.index = later.index
    same = (later[val.PAYLOAD_COLUMNS] == original[val.PAYLOAD_COLUMNS]).all(axis=1)
    reason[later.index] = same.map({True: "DUPLICATE_EXACT", False: "DUPLICATE_CONFLICT"})
    detail[later.index] = ("duplicate of " + original["source_file"]
                           + " line " + original["line_no"].astype(str))


current_parse, current_flag = val.parse_timestamps, val._flag_duplicates
variants = [
    ("original: per-value parse, sort every row for duplicates", per_value_parse, sort_everything_flag_duplicates),
    ("bulk parse with strptime format", strptime_parse, sort_everything_flag_duplicates),
    ("bulk parse with ISO8601 format", current_parse, sort_everything_flag_duplicates),
    ("+ duplicates: sort only repeated event_ids (current code)", current_parse, current_flag),
]

results, reference = [], None
for label, parse, flag in variants:
    val.parse_timestamps, val._flag_duplicates = parse, flag
    try:
        out = val.validate(raw, users, cfg)
        seconds = best_of(lambda: val.validate(raw, users, cfg))
    finally:
        val.parse_timestamps, val._flag_duplicates = current_parse, current_flag
    if reference is None:
        reference = out
    else:
        pd.testing.assert_frame_equal(reference[0], out[0])
        pd.testing.assert_frame_equal(reference[1], out[1])
    results.append((label, seconds))

table = pd.DataFrame(results, columns=["variant", "validate_s"])
table["vs_original"] = (table.validate_s.iloc[0] / table.validate_s).round(2)
print(table.round(3).to_string(index=False))
print("outputs identical across all variants: True")

                                                  variant  validate_s  vs_original
 original: per-value parse, sort every row for duplicates       1.965         1.00
                          bulk parse with strptime format       1.730         1.14
                           bulk parse with ISO8601 format       1.515         1.30
+ duplicates: sort only repeated event_ids (current code)       1.377         1.43
outputs identical across all variants: True


## 9. Package the results

In [53]:
!python build_notebook.py --zip sessionization_project.zip
!unzip -l sessionization_project.zip | tail -n 25

wrote sessionization_project.zip: 40 files
     3891  2026-11-01 13:09   tests/test_ingest.py
     1980  2026-11-01 13:09   tests/test_notebook.py
     6691  2026-11-01 13:09   tests/test_publish.py
     4138  2026-11-01 13:09   tests/test_reports.py
     4628  2026-11-01 13:09   tests/test_run.py
     2566  2026-11-01 13:09   tests/test_sample_data.py
     5801  2026-11-01 13:09   tests/test_sessionize.py
     3425  2026-11-01 13:09   tests/test_timestamps.py
     5751  2026-11-01 13:09   tests/test_validate.py
     3751  2026-11-01 13:09   build_notebook.py
     7411  2026-11-01 13:09   notebook_demo.py
      161  2026-11-01 13:09   data/ref/users.csv
      519  2026-11-01 13:09   data/raw/events_a.csv
      542  2026-11-01 13:09   data/raw/events_b.csv
      375  2026-11-01 13:09   data/expected/expected_rejects.csv
      140  2026-11-01 13:09   data/expected/expected_run_summary.json
      453  2026-11-01 13:09   data/expected/expected_sessions.csv
     9275  2026-11-01 13:09   REA

In [54]:
try:
    from google.colab import files
    files.download("sessionization_project.zip")
except Exception as exc:  # not running in the Colab front end
    print("download skipped:", exc)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>